# Held-out five-arm ablation, all four personas

Generated by `notebooks/build_eval_ablation.py`. Do not hand-edit. Change the repository
source or `configs/eval_ablation.yaml`, then rebuild.

Every held-out question is answered five times per persona, once per rung of the ladder:

| arm | what it adds |
| --- | --- |
| `base_embedder_no_profile` | naive RAG, no persona anywhere |
| `base_embedder` | persona in the retrieval instruction and the answer prompt |
| `trained_embedder` | the ROPG-KD retriever adapter |
| `trained_embedder_base_rewriter` | query rewriting by an *untrained* Qwen3-4B |
| `trained_embedder_rewriter` | the DPO-trained rewriter adapter |

Two judges score every row. `gpt-5.6-luna` labelled the DPO preference pairs and was the
ROPG teacher, so `gemini-3.7-flash` scores the identical rubric alongside it and
`judge_agreement.csv` reports how far the two agree.

Run order, one non-interactive pass, top to bottom: packages, secrets, runtime knobs,
sources, adapter discovery, preflight, two-question smoke, full run, inspection, export.

Only the runtime-knobs cell is meant to be edited. Everything else, including the number of
rows the run expects, is derived from the config.


In [ ]:
!pip install -q "sentence-transformers==5.6.0" "peft==0.18.0" "transformers==4.57.6" "accelerate==1.14.0" "unsloth==2026.8.22" "openai" "google-genai==2.20.0"
!pip install -q faiss-gpu || pip install -q faiss-cpu
!pip uninstall -q -y torchao

## Runtime knobs — the only cell to edit

In [ ]:
# Every override defaults to None, meaning "use the embedded YAML".
DATA_ROOT = "/kaggle/input/datasets/alirezahsn/simurgh-data"
ROPG_ADAPTER_DIR = None      # set to a path to skip discovery for the ROPG checkpoint
DPO_ADAPTER_DIR = None       # set to a path to skip discovery for the DPO adapter
REPLICATES_OVERRIDE = None   # e.g. [1, 2, 3] to repeat every remote call three times
PERSONAS_OVERRIDE = None     # e.g. ["newcomer"] to score the held-out persona only
MAX_WORKERS_OVERRIDE = None  # e.g. 4 if the endpoint rate-limits
SMOKE_LIMIT = 2              # questions in the smoke run

# Replicates are repetitions of identical remote calls, not seeds: they measure endpoint
# sampling variance and nothing else. One replicate makes no variance claim at all.


## Secrets and paths

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Set before anything imports data.settings, which reads the environment once at import.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for key in ("OPENAI_API_KEY", "OPENAI_BASE_URL", "GEMINI_API_KEY", "GEMINI_ENDPOINT"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception as error:
        print(f"secret {key} unavailable: {error}")
for key in ("OPENAI_API_KEY", "OPENAI_BASE_URL", "GEMINI_API_KEY", "GEMINI_ENDPOINT"):
    print(f"{key} present: {bool(os.environ.get(key))}")

DATA_ROOT = Path(DATA_ROOT)
OUTPUT_ROOT = Path("/kaggle/working/eval")
WORKDIR = Path("/kaggle/working/eval_job")
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
os.environ["PYTHONPATH"] = str(WORKDIR / "src")
print("cwd:", WORKDIR, "| data:", DATA_ROOT)


## Materialize the exact repository contract

In [ ]:
import json

SOURCE_FILES = json.loads('{"benchmarks/compare_runs.py": "\\"\\"\\"Paired significance testing over the per-query vectors in ``training_log.json``.\\n\\n``rl.ropg_kd.evaluate_retrieval`` returns per-query metric vectors and the training\\nloop persists them for the untrained baseline and every epoch, precisely so that two\\nevaluations can be compared with a **paired** test. This is the consumer. Until it\\nruns, no delta in the results tables is claimable: with *n* = 92 per persona the\\ndifferences at stake are about one standard error, and the untrained baseline\'s own\\nper-persona nDCG@5 spread (0.5368 / 0.5591) already sits under one — before any\\ngradient is taken.\\n\\nPairing is what buys the power. Every evaluation scores the *same* 276 (question,\\npersona) rows, so per-query difficulty — by far the largest source of variance —\\ncancels in the difference. An unpaired comparison of two means throws that away.\\n\\nTwo statistics, because they answer different questions:\\n\\n* **Bootstrap percentile CI** on the mean difference — the effect size and its\\n  uncertainty. Resamples queries with replacement.\\n* **Sign-flip permutation p-value** — the significance. Under the null \\"the paired\\n  differences are symmetric about zero\\", flipping their signs at random is exactly\\n  as likely as what was observed. This is a better-calibrated p-value than reading\\n  one off the bootstrap distribution, and costs the same.\\n\\nReported p-values are also **Holm-corrected** across the metric family, because a\\nrun reports a dozen metrics and the largest of twelve noise draws looks impressive\\non its own.\\n\\nUsage::\\n\\n    # untrained baseline (epoch 0) vs the run\'s best epoch\\n    uv run python benchmarks/compare_runs.py runB_training_log.json\\n\\n    # one arm against another, at each one\'s best epoch\\n    uv run python benchmarks/compare_runs.py runA_log.json runC_log.json\\n\\n    # pin specific epochs, and write the table into the thesis\\n    uv run python benchmarks/compare_runs.py runB.json --b-epoch 1 \\\\\\\\\\n        --out docs/results/stage1-runB-epoch1.md\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\n\\n# Not per-query, so not paired-testable: one scalar per epoch. Reported for context\\n# only. It is also the one number that is NOT comparable across arms — a filtered\\n# arm computes its val loss over a different (smaller) set of triplets.\\nSCALAR_KEYS = (\\"val_loss\\", \\"train_loss\\", \\"epoch\\")\\n\\n\\ndef load_log(path: str | Path) -> dict[str, Any]:\\n    with open(path, encoding=\\"utf-8\\") as fh:\\n        log = json.load(fh)\\n    if \\"epoch_metrics\\" not in log:\\n        raise ValueError(f\\"{path}: not a training_log.json (no \'epoch_metrics\' key)\\")\\n    return log\\n\\n\\ndef pick_entry(log: dict[str, Any], path: str, epoch: int | None) -> dict[str, Any]:\\n    \\"\\"\\"Return the epoch entry to compare, defaulting to the run\'s own best epoch.\\"\\"\\"\\n    entries = log[\\"epoch_metrics\\"]\\n    want = log.get(\\"best_epoch\\", 0) if epoch is None else epoch\\n    for entry in entries:\\n        if entry.get(\\"epoch\\") == want:\\n            if \\"per_query\\" not in entry:\\n                raise ValueError(\\n                    f\\"{path}: epoch {want} has no per_query vectors. Retrieval eval was \\"\\n                    \\"disabled for that run (no corpus, or no val groups), so there is \\"\\n                    \\"nothing to pair.\\"\\n                )\\n            return entry\\n    have = [e.get(\\"epoch\\") for e in entries]\\n    raise ValueError(f\\"{path}: no epoch {want} in the log (have {have})\\")\\n\\n\\ndef check_alignment(a: dict[str, Any], b: dict[str, Any], label_a: str, label_b: str) -> None:\\n    \\"\\"\\"Refuse to pair two evaluations that did not score the same rows in the same order.\\n\\n    ``evaluate_retrieval`` builds its vectors in group order and skips a group only\\n    when none of its judged chunks are in the corpus — a decision that depends on the\\n    data alone, never on the model. So identical ``persona_ids`` is a sound proxy for\\n    \\"same rows, same order\\". If it fails, the two runs used different val data and the\\n    pairing would silently compare unrelated questions.\\n    \\"\\"\\"\\n    pa, pb = a[\\"per_query\\"][\\"persona_ids\\"], b[\\"per_query\\"][\\"persona_ids\\"]\\n    if len(pa) != len(pb):\\n        raise ValueError(\\n            f\\"{label_a} scored {len(pa)} queries but {label_b} scored {len(pb)}. \\"\\n            \\"These runs did not use the same val set — pairing is invalid.\\"\\n        )\\n    if pa != pb:\\n        n_diff = sum(1 for x, y in zip(pa, pb, strict=True) if x != y)\\n        raise ValueError(\\n            f\\"{label_a} and {label_b} disagree on persona_ids at {n_diff}/{len(pa)} \\"\\n            \\"positions. Same length but different rows or a different order — pairing \\"\\n            \\"is invalid.\\"\\n        )\\n\\n\\ndef paired_stats(\\n    before: np.ndarray, after: np.ndarray, n_boot: int, rng: np.random.Generator\\n) -> dict[str, float]:\\n    \\"\\"\\"Mean paired difference with a bootstrap CI and a sign-flip permutation p-value.\\"\\"\\"\\n    d = after - before\\n    n = d.size\\n    observed = float(d.mean())\\n\\n    idx = rng.integers(0, n, size=(n_boot, n))\\n    boot = d[idx].mean(axis=1)\\n    lo, hi = np.percentile(boot, [2.5, 97.5])\\n\\n    # Sign-flip permutation. The +1 in both terms is the standard guard against\\n    # reporting p = 0 from a finite number of permutations.\\n    signs = rng.choice(np.array([-1.0, 1.0]), size=(n_boot, n))\\n    null = (d * signs).mean(axis=1)\\n    p = (np.sum(np.abs(null) >= abs(observed)) + 1) / (n_boot + 1)\\n\\n    return {\\n        \\"before\\": float(before.mean()),\\n        \\"after\\": float(after.mean()),\\n        \\"delta\\": observed,\\n        \\"lo\\": float(lo),\\n        \\"hi\\": float(hi),\\n        \\"p\\": float(p),\\n        \\"n\\": n,\\n    }\\n\\n\\ndef holm(pvals: list[float]) -> list[float]:\\n    \\"\\"\\"Holm-Bonferroni step-down adjustment. Controls family-wise error across metrics.\\n\\n    Uniformly more powerful than plain Bonferroni at the same guarantee, and it makes\\n    no independence assumption — which matters here, where nDCG@1..5 and Hit@1..5 are\\n    heavily correlated by construction.\\n    \\"\\"\\"\\n    m = len(pvals)\\n    order = sorted(range(m), key=lambda i: pvals[i])\\n    adjusted = [0.0] * m\\n    running = 0.0\\n    for rank, i in enumerate(order):\\n        running = max(running, min(1.0, (m - rank) * pvals[i]))\\n        adjusted[i] = running\\n    return adjusted\\n\\n\\ndef stars(p: float) -> str:\\n    return \\"***\\" if p < 0.001 else \\"**\\" if p < 0.01 else \\"*\\" if p < 0.05 else \\"\\"\\n\\n\\ndef compare(\\n    entry_a: dict[str, Any],\\n    entry_b: dict[str, Any],\\n    subset: str | None,\\n    n_boot: int,\\n    seed: int,\\n) -> list[tuple[str, dict[str, float]]]:\\n    \\"\\"\\"Paired stats for every metric both entries carry, optionally within one persona.\\"\\"\\"\\n    pq_a, pq_b = entry_a[\\"per_query\\"], entry_b[\\"per_query\\"]\\n    metrics = [k for k in pq_a if k != \\"persona_ids\\" and k in pq_b]\\n\\n    mask = None\\n    if subset is not None:\\n        mask = np.array([pid == subset for pid in pq_a[\\"persona_ids\\"]])\\n        if not mask.any():\\n            return []\\n\\n    rows = []\\n    for name in metrics:\\n        # One generator per metric, seeded identically, so a metric\'s numbers do not\\n        # shift when an unrelated metric is added to the log.\\n        rng = np.random.default_rng(seed)\\n        before = np.asarray(pq_a[name], dtype=float)\\n        after = np.asarray(pq_b[name], dtype=float)\\n        if mask is not None:\\n            before, after = before[mask], after[mask]\\n        rows.append((name, paired_stats(before, after, n_boot, rng)))\\n\\n    adjusted = holm([s[\\"p\\"] for _, s in rows])\\n    for (_, s), padj in zip(rows, adjusted, strict=True):\\n        s[\\"p_holm\\"] = padj\\n    return rows\\n\\n\\ndef render_table(rows: list[tuple[str, dict[str, float]]], label_a: str, label_b: str) -> str:\\n    if not rows:\\n        return \\"_(no overlapping metrics)_\\\\n\\"\\n    out = [\\n        f\\"| Metric | {label_a} | {label_b} | Δ | 95% CI | p | p (Holm) | |\\",\\n        \\"|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for name, s in rows:\\n        out.append(\\n            f\\"| `{name}` | {s[\'before\']:.4f} | {s[\'after\']:.4f} | {s[\'delta\']:+.4f} | \\"\\n            f\\"[{s[\'lo\']:+.4f}, {s[\'hi\']:+.4f}] | {s[\'p\']:.4f} | {s[\'p_holm\']:.4f} | \\"\\n            f\\"{stars(s[\'p_holm\'])} |\\"\\n        )\\n    return \\"\\\\n\\".join(out) + \\"\\\\n\\"\\n\\n\\ndef describe(log: dict[str, Any], entry: dict[str, Any], path: str) -> str:\\n    cfg = log.get(\\"config\\", {})\\n    anchor = cfg.get(\\"anchor\\", {}) or {}\\n    epoch = entry.get(\\"epoch\\")\\n    tag = \\"baseline (untrained)\\" if epoch == 0 else f\\"epoch {epoch}\\"\\n    return (\\n        f\\"`{Path(path).name}` {tag} — mode={cfg.get(\'mode\')}, \\"\\n        f\\"anchor={anchor.get(\'mode\', \'none\')}, \\"\\n        f\\"data={Path(str(cfg.get(\'data\', {}).get(\'train_data\', \'?\'))).name}, \\"\\n        f\\"seed={log.get(\'seed\')}\\"\\n    )\\n\\n\\ndef main() -> int:\\n    ap = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    ap.add_argument(\\n        \\"logs\\", nargs=\\"+\\", help=\\"one training_log.json (epoch 0 vs best), or two (arm vs arm)\\"\\n    )\\n    ap.add_argument(\\n        \\"--a-epoch\\",\\n        type=int,\\n        default=None,\\n        help=\\"epoch from the first log (default: 0 for one log, best for two)\\",\\n    )\\n    ap.add_argument(\\n        \\"--b-epoch\\",\\n        type=int,\\n        default=None,\\n        help=\\"epoch from the second log (default: that run\'s best)\\",\\n    )\\n    ap.add_argument(\\n        \\"--n-boot\\",\\n        type=int,\\n        default=10000,\\n        help=\\"bootstrap / permutation resamples (default 10000)\\",\\n    )\\n    ap.add_argument(\\"--seed\\", type=int, default=42)\\n    ap.add_argument(\\n        \\"--swap\\",\\n        action=\\"store_true\\",\\n        help=\\"pair persona-matched retrieval against the rotated-persona control \\"\\n        \\"within one log (needs eval.persona_swap enabled for that run)\\",\\n    )\\n    ap.add_argument(\\"--no-personas\\", action=\\"store_true\\", help=\\"skip the per-persona breakdown\\")\\n    ap.add_argument(\\"--out\\", type=Path, default=None, help=\\"also write the Markdown to this path\\")\\n    args = ap.parse_args()\\n\\n    if len(args.logs) > 2:\\n        ap.error(\\"compare at most two logs\\")\\n\\n    if args.swap and len(args.logs) != 1:\\n        ap.error(\\"--swap compares matched vs swapped inside ONE log; pass a single log\\")\\n\\n    single = len(args.logs) == 1\\n    path_a = args.logs[0]\\n    path_b = args.logs[0] if single else args.logs[1]\\n    log_a = load_log(path_a)\\n    log_b = log_a if single else load_log(path_b)\\n\\n    # One log means \\"did training beat the untrained encoder?\\", which is the question\\n    # the epoch-0 row exists to answer. Two logs means \\"did arm B beat arm A?\\".\\n    a_epoch = args.a_epoch if args.a_epoch is not None else (0 if single else None)\\n    entry_a = pick_entry(log_a, path_a, a_epoch)\\n    entry_b = pick_entry(log_b, path_b, args.b_epoch)\\n\\n    if entry_a is entry_b:\\n        ap.error(\\"the two selected entries are the same epoch of the same log\\")\\n\\n    if args.swap:\\n        # Matched vs rotated-persona, same epoch, same rows, same corpus embedding. The\\n        # delta is the personalisation signal in isolation: everything except the\\n        # `Instruct:` prefix is held fixed, so a null result here means the encoder is\\n        # not reading the persona at all and every headline gain is generic retrieval.\\n        entry_a = pick_entry(log_a, path_a, args.a_epoch if args.a_epoch is not None else None)\\n        swap = entry_a.get(\\"persona_swap\\")\\n        if not swap:\\n            raise ValueError(\\n                f\\"{path_a}: epoch {entry_a.get(\'epoch\')} has no persona_swap block. That \\"\\n                \\"run predates the control, or was configured with eval.persona_swap: false.\\"\\n            )\\n        entry_b = {\\"epoch\\": entry_a.get(\\"epoch\\"), \\"per_query\\": swap[\\"per_query\\"]}\\n        label_a = describe(log_a, entry_a, path_a) + \\" — persona-MATCHED\\"\\n        label_b = describe(log_a, entry_a, path_a) + \\" — persona-SWAPPED (rotated)\\"\\n    else:\\n        label_a = describe(log_a, entry_a, path_a)\\n        label_b = describe(log_b, entry_b, path_b)\\n    check_alignment(entry_a, entry_b, label_a, label_b)\\n\\n    lines = [\\n        \\"# Paired comparison\\",\\n        \\"\\",\\n        f\\"- **A:** {label_a}\\",\\n        f\\"- **B:** {label_b}\\",\\n        f\\"- {entry_a[\'per_query\'][\'persona_ids\'].__len__()} paired queries, \\"\\n        f\\"{args.n_boot} resamples, seed {args.seed}\\",\\n        \\"- Δ = B − A. CI is a bootstrap percentile interval; p is a sign-flip permutation\\",\\n        \\"  test, Holm-corrected across the metric family. Stars use the Holm value.\\",\\n        \\"\\",\\n    ]\\n\\n    if not single and not args.swap:\\n        # Two arms trained on the same val set must agree exactly at epoch 0 — the\\n        # adapter is the identity there. A mismatch means something leaked into the\\n        # eval path (different corpus, different val file, changed relevance rule),\\n        # and every downstream comparison between these two runs is void.\\n        try:\\n            base_a = pick_entry(log_a, path_a, 0)\\n            base_b = pick_entry(log_b, path_b, 0)\\n            same = base_a[\\"per_query\\"] == base_b[\\"per_query\\"]\\n            lines.append(\\n                \\"- **Baseline parity:** epoch-0 vectors are identical ✅\\"\\n                if same\\n                else \\"- **Baseline parity: FAILED ❌** — the two runs\' untrained epoch-0 \\"\\n                \\"evaluations differ, so their eval paths are not the same. Do not \\"\\n                \\"compare these runs until that is explained.\\"\\n            )\\n            lines.append(\\"\\")\\n        except ValueError as exc:\\n            lines += [f\\"- Baseline parity: not checkable ({exc})\\", \\"\\"]\\n\\n    for key in SCALAR_KEYS:\\n        if not args.swap and key in entry_a and key in entry_b and key != \\"epoch\\":\\n            note = \\"\\"\\n            if key == \\"val_loss\\" and not single:\\n                data_a = Path(str(log_a.get(\\"config\\", {}).get(\\"data\\", {}).get(\\"train_data\\", \\"\\"))).name\\n                data_b = Path(str(log_b.get(\\"config\\", {}).get(\\"data\\", {}).get(\\"train_data\\", \\"\\"))).name\\n                if data_a != data_b:\\n                    note = \\" — **not comparable across arms** (the runs score different val sets)\\"\\n            lines.append(f\\"- `{key}`: {entry_a[key]:.4f} → {entry_b[key]:.4f}{note}\\")\\n    lines.append(\\"\\")\\n\\n    lines += [\\n        \\"## Overall\\",\\n        \\"\\",\\n        render_table(compare(entry_a, entry_b, None, args.n_boot, args.seed), \\"A\\", \\"B\\"),\\n    ]\\n\\n    if not args.no_personas:\\n        for persona in sorted(set(entry_a[\\"per_query\\"][\\"persona_ids\\"])):\\n            rows = compare(entry_a, entry_b, persona, args.n_boot, args.seed)\\n            n = rows[0][1][\\"n\\"] if rows else 0\\n            lines += [f\\"## Persona: {persona} (n = {n})\\", \\"\\", render_table(rows, \\"A\\", \\"B\\")]\\n\\n    report = \\"\\\\n\\".join(lines)\\n    print(report)\\n    if args.out:\\n        args.out.parent.mkdir(parents=True, exist_ok=True)\\n        args.out.write_text(report, encoding=\\"utf-8\\")\\n        print(f\\"\\\\nwrote {args.out}\\", file=sys.stderr)\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    try:\\n        raise SystemExit(main())\\n    except ValueError as exc:\\n        # These are the alignment and provenance guards, not bugs. A traceback would\\n        # bury the one sentence that says which runs cannot be compared and why.\\n        print(f\\"error: {exc}\\", file=sys.stderr)\\n        raise SystemExit(1) from None\\n", "benchmarks/eval_runner.py": "\\"\\"\\"Five-arm held-out evaluation across every learner persona, on two GPUs.\\n\\nOne worker process per GPU, launched by torchrun. Each rank owns a disjoint slice of the\\n``(question, persona)`` cross product and writes its own append-only JSONL; rank 0 merges\\nafter a single barrier. There is no shared mutable state, no index on disk, and no\\ncollective beyond that barrier, which is why the process group is gloo rather than NCCL.\\n\\nThe arms form a ladder, each rung isolating one intervention:\\n\\n===============================  ==================================================\\nbase_embedder_no_profile         naive RAG: no persona anywhere in the pipeline\\nbase_embedder                    + persona in the retrieval instruction and prompt\\ntrained_embedder                 + the ROPG-KD retriever adapter\\ntrained_embedder_base_rewriter   + query rewriting by an *untrained* Qwen3-4B\\ntrained_embedder_rewriter        + the DPO-trained rewriter adapter\\n===============================  ==================================================\\n\\nThe fourth rung is what makes the fifth interpretable. ``docs/results/dpo-arms-seed42-v1.md``\\nreports the DPO rewriter at Holm *p* = 0.502 against its own base model, so a delta measured\\nagainst \\"no rewriter at all\\" would credit the adapter for the act of rewriting.\\n\\nThe index is built from raw ``corpus.jsonl`` text, never through ``rag.dense_store``: that\\nmodule normalizes Persian text and all 171 corpus chunks change under it, while\\n``rl.ropg_kd.load_corpus`` trained the adapter on the raw strings. Retrieving over\\nnormalized text would score the adapter on inputs it never saw.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport csv\\nimport gc\\nimport hashlib\\nimport json\\nimport os\\nimport sys\\nimport threading\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import dataclass\\nfrom datetime import UTC, datetime, timedelta\\nfrom pathlib import Path\\nfrom typing import TYPE_CHECKING, Any\\n\\nimport numpy as np\\nimport yaml\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nSRC_DIR = PROJECT_ROOT / \\"src\\"\\nBENCHMARKS_DIR = PROJECT_ROOT / \\"benchmarks\\"\\nfor _p in (str(SRC_DIR), str(BENCHMARKS_DIR)):\\n    if _p not in sys.path:\\n        sys.path.insert(0, _p)\\n\\n# ruff: noqa: E402  (deliberate mid-file imports; sys.path must be set first)\\nfrom compare_runs import holm, paired_stats\\nfrom data.questions import load_question\\nfrom data.settings import GEMINI_API_KEY, GEMINI_ENDPOINT, OPENAI_API_KEY, OPENAI_BASE_URL\\nfrom judge import SCORE_FIELDS, GeminiJudge, JudgeError, LunaJudge, RetryPolicy\\nfrom personalization.profiles import PERSONAS, render_profile\\n\\nif TYPE_CHECKING:\\n    from collections.abc import Callable, Iterable\\n\\nARM_NAMES = (\\n    \\"base_embedder_no_profile\\",\\n    \\"base_embedder\\",\\n    \\"trained_embedder\\",\\n    \\"trained_embedder_base_rewriter\\",\\n    \\"trained_embedder_rewriter\\",\\n)\\nEMBEDDER_KINDS = (\\"base\\", \\"ropg\\")\\nREWRITER_KINDS = (\\"none\\", \\"base\\", \\"dpo\\")\\nJUDGE_ROLES = (\\"primary\\", \\"secondary\\")\\n\\n#: Each rung against the one below it. Adjacent rungs differ by exactly one intervention,\\n#: which is the only reason a delta can be attributed to that intervention.\\nCOMPARISONS = (\\n    (\\"base_embedder - base_embedder_no_profile\\", \\"base_embedder_no_profile\\", \\"base_embedder\\"),\\n    (\\"trained_embedder - base_embedder\\", \\"base_embedder\\", \\"trained_embedder\\"),\\n    (\\n        \\"trained_embedder_base_rewriter - trained_embedder\\",\\n        \\"trained_embedder\\",\\n        \\"trained_embedder_base_rewriter\\",\\n    ),\\n    (\\n        \\"trained_embedder_rewriter - trained_embedder_base_rewriter\\",\\n        \\"trained_embedder_base_rewriter\\",\\n        \\"trained_embedder_rewriter\\",\\n    ),\\n)\\n\\nANSWER_SYSTEM = (\\n    \\"You are a Persian-language educational assistant helping a ninth-grade student. \\"\\n    \\"Answer using only the numbered passages you are given. If they do not contain the \\"\\n    \\"answer, say so honestly instead of guessing. Always reply in Persian. Cite the \\"\\n    \\"passages you used by their number, like [1].\\"\\n)\\nANSWER_SYSTEM_PERSONALIZED = (\\n    ANSWER_SYSTEM\\n    + \\" Adapt the depth, vocabulary, and style of your explanation to the learner profile \\"\\n    \\"you are given.\\"\\n)\\n\\n\\nclass RemoteCallError(RuntimeError):\\n    \\"\\"\\"A remote generator call exhausted its retries.\\"\\"\\"\\n\\n\\n@dataclass(frozen=True)\\nclass RetrievedChunk:\\n    \\"\\"\\"One retrieved corpus chunk.\\n\\n    Not ``rag.store.Hit``: that type\'s ``source`` and ``chunk_index`` mean file-of-origin\\n    and within-file ordinal, so a corpus ``chunk_id`` would have to be smuggled through a\\n    field that means something else — and importing ``rag.store`` drags hazm and sqlite3\\n    into a job that needs neither.\\n    \\"\\"\\"\\n\\n    chunk_id: str\\n    text: str\\n    score: float\\n    rank: int\\n\\n    def as_dict(self) -> dict[str, Any]:\\n        return {\\n            \\"chunk_id\\": self.chunk_id,\\n            \\"text\\": self.text,\\n            \\"score\\": self.score,\\n            \\"rank\\": self.rank,\\n        }\\n\\n\\n@dataclass(frozen=True)\\nclass Case:\\n    \\"\\"\\"One ``(question, persona)`` pair: the unit of sharding and of GPU work.\\"\\"\\"\\n\\n    question_ref: str\\n    persona_id: str\\n    query: str\\n    gold_answer: str\\n    gold_explanation: str\\n    profile_rendered: str\\n\\n\\n@dataclass(frozen=True)\\nclass ArmContext:\\n    \\"\\"\\"What one arm retrieved for one case, and the query it retrieved with.\\"\\"\\"\\n\\n    retrieval_query: str\\n    retrieval_instruction: str\\n    chunks: list[RetrievedChunk]\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Config, digests, inputs\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef load_config(path: str | Path) -> dict[str, Any]:\\n    with open(path, encoding=\\"utf-8\\") as handle:\\n        config = yaml.safe_load(handle)\\n    if not isinstance(config, dict):\\n        raise ValueError(f\\"{path}: config must be a mapping\\")\\n    return config\\n\\n\\ndef config_digest(config: dict[str, Any]) -> str:\\n    \\"\\"\\"SHA-256 over the *resolved* config, which is what a record\'s provenance means.\\"\\"\\"\\n    payload = json.dumps(config, sort_keys=True, ensure_ascii=False)\\n    return hashlib.sha256(payload.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with open(path, \\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef read_test_qids(path: str | Path) -> list[str]:\\n    lines = Path(path).read_text(encoding=\\"utf-8\\").splitlines()\\n    return [line.strip() for line in lines if line.strip()]\\n\\n\\ndef load_corpus(path: str | Path) -> list[dict[str, Any]]:\\n    rows = []\\n    with open(path, encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            line = line.strip()\\n            if line:\\n                rows.append(json.loads(line))\\n    return rows\\n\\n\\ndef build_cases(config: dict[str, Any], limit: int | None = None) -> list[Case]:\\n    \\"\\"\\"Ordered ``(question, persona)`` cross product — the stable sharding order.\\n\\n    ``--limit`` truncates the *question* list before the cross product, so a limited run\\n    still covers every persona.\\n    \\"\\"\\"\\n    questions_dir = Path(config[\\"data\\"][\\"questions_dir\\"])\\n    refs = read_test_qids(config[\\"data\\"][\\"test_qids_path\\"])\\n    if limit is not None:\\n        refs = refs[:limit]\\n    cases: list[Case] = []\\n    for ref in refs:\\n        exam_stem, qid = ref.split(\\":\\", 1)\\n        context = load_question(exam_stem, qid, questions_dir)\\n        answer = \\"\\" if context.answer is None else str(context.answer)\\n        explanation = context.explanation or \\"\\"\\n        for persona_id in config[\\"personas\\"]:\\n            cases.append(\\n                Case(\\n                    question_ref=ref,\\n                    persona_id=persona_id,\\n                    query=context.query,\\n                    gold_answer=answer,\\n                    gold_explanation=explanation,\\n                    profile_rendered=render_profile(persona_id),\\n                )\\n            )\\n    return cases\\n\\n\\ndef build_expected_keys(\\n    config: dict[str, Any], cases: Iterable[Case]\\n) -> list[tuple[int, str, str, str]]:\\n    \\"\\"\\"Every ``(replicate, arm, question_ref, persona_id)`` the run must produce.\\"\\"\\"\\n    arm_names = [arm[\\"name\\"] for arm in config[\\"arms\\"]]\\n    keys = [\\n        (replicate, arm_name, case.question_ref, case.persona_id)\\n        for replicate in config[\\"replicates\\"]\\n        for arm_name in arm_names\\n        for case in cases\\n    ]\\n    if len(set(keys)) != len(keys):\\n        raise ValueError(\\"expected keys are not unique; check personas and replicates\\")\\n    return keys\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Validation and preflight\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef _adapter_base_model(adapter_dir: Path) -> str | None:\\n    config_path = adapter_dir / \\"adapter_config.json\\"\\n    if not config_path.is_file():\\n        return None\\n    try:\\n        return json.loads(config_path.read_text(encoding=\\"utf-8\\")).get(\\"base_model_name_or_path\\")\\n    except json.JSONDecodeError:\\n        return None\\n\\n\\ndef probe_models(config: dict[str, Any]) -> dict[str, bool]:\\n    \\"\\"\\"One minimal call per remote model literal.\\n\\n    A misspelled model name is otherwise discovered after the GPU stages, hours in.\\n    \\"\\"\\"\\n    from rag.llm import GeminiClient, OpenAICompatClient\\n\\n    results: dict[str, bool] = {}\\n    generator = config[\\"generator\\"][\\"model\\"]\\n    primary = config[\\"judges\\"][\\"primary\\"][\\"model\\"]\\n    secondary = config[\\"judges\\"][\\"secondary\\"][\\"model\\"]\\n    for model in (generator, primary):\\n        client = OpenAICompatClient(\\n            base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY, model=model, max_tokens=16\\n        )\\n        results[model] = bool(client.chat([{\\"role\\": \\"user\\", \\"content\\": \\"ping\\"}]).strip())\\n    gemini = GeminiClient(\\n        base_url=GEMINI_ENDPOINT,\\n        api_key=GEMINI_API_KEY,\\n        model=secondary,\\n        temperature=0.0,\\n        max_output_tokens=16,\\n        thinking_level=config[\\"judges\\"][\\"secondary\\"].get(\\"thinking_level\\"),\\n    )\\n    results[secondary] = bool(gemini.generate(\\"ping\\").strip())\\n    return results\\n\\n\\ndef validate(config: dict[str, Any], *, probe: bool = True) -> dict[str, Any]:\\n    \\"\\"\\"Check everything checkable before a single byte of model weight is loaded.\\n\\n    Collects every failure rather than raising on the first: a preflight that reports one\\n    problem per run costs one Kaggle session per problem.\\n    \\"\\"\\"\\n    errors: list[str] = []\\n    report: dict[str, Any] = {}\\n\\n    questions_dir = Path(config[\\"data\\"][\\"questions_dir\\"])\\n    qids_path = Path(config[\\"data\\"][\\"test_qids_path\\"])\\n    if not qids_path.is_file():\\n        errors.append(f\\"test_qids_path not found: {qids_path}\\")\\n        refs: list[str] = []\\n    else:\\n        refs = read_test_qids(qids_path)\\n        if len(set(refs)) != len(refs):\\n            errors.append(\\"test_qids contains duplicate entries\\")\\n        for ref in refs:\\n            if ref.count(\\":\\") != 1 or not all(part.strip() for part in ref.split(\\":\\")):\\n                errors.append(f\\"test_qid {ref!r} is not in exam_stem:qid form\\")\\n                continue\\n            exam_stem, qid = ref.split(\\":\\", 1)\\n            try:\\n                load_question(exam_stem, qid, questions_dir)\\n            except (FileNotFoundError, KeyError) as exc:\\n                errors.append(f\\"test_qid {ref!r} does not load: {exc}\\")\\n    report[\\"n_questions\\"] = len(refs)\\n\\n    personas = config.get(\\"personas\\")\\n    if not isinstance(personas, list) or not personas:\\n        errors.append(\\"personas must be a nonempty list\\")\\n        personas = []\\n    elif len(set(personas)) != len(personas):\\n        errors.append(\\"personas contains duplicates\\")\\n    unknown = [p for p in personas if p not in PERSONAS]\\n    if unknown:\\n        errors.append(f\\"unknown personas {unknown}; valid: {sorted(PERSONAS)}\\")\\n    report[\\"personas\\"] = [{\\"id\\": p, \\"split\\": PERSONAS[p].split} for p in personas if p in PERSONAS]\\n\\n    replicates = config.get(\\"replicates\\")\\n    if not isinstance(replicates, list) or not replicates:\\n        errors.append(\\"replicates must be a nonempty list\\")\\n        replicates = []\\n    else:\\n        if len(set(replicates)) != len(replicates):\\n            errors.append(\\"replicates contains duplicates\\")\\n        bad = [r for r in replicates if not isinstance(r, int) or isinstance(r, bool) or r < 1]\\n        if bad:\\n            errors.append(f\\"replicates must be positive integers; got {bad}\\")\\n    report[\\"replicates\\"] = list(replicates)\\n\\n    arms = config.get(\\"arms\\") or []\\n    names = [arm.get(\\"name\\") for arm in arms]\\n    if names != list(ARM_NAMES):\\n        errors.append(f\\"arms must be exactly {list(ARM_NAMES)} in order; got {names}\\")\\n    for arm in arms:\\n        if arm.get(\\"embedder\\") not in EMBEDDER_KINDS:\\n            errors.append(f\\"arm {arm.get(\'name\')!r}: embedder must be one of {EMBEDDER_KINDS}\\")\\n        if arm.get(\\"rewriter\\") not in REWRITER_KINDS:\\n            errors.append(f\\"arm {arm.get(\'name\')!r}: rewriter must be one of {REWRITER_KINDS}\\")\\n        if not isinstance(arm.get(\\"profile\\"), bool):\\n            errors.append(f\\"arm {arm.get(\'name\')!r}: profile must be a boolean\\")\\n    n_unprofiled = sum(1 for arm in arms if arm.get(\\"profile\\") is False)\\n    if n_unprofiled != 1:\\n        errors.append(f\\"exactly one arm must set profile: false; found {n_unprofiled}\\")\\n    report[\\"arms\\"] = names\\n\\n    adapters = {\\n        \\"ropg_adapter_path\\": (\\"Qwen3-Embedding-0.6B\\", config[\\"artifacts\\"][\\"ropg_adapter_path\\"]),\\n        \\"dpo_adapter_path\\": (\\"Qwen3-4B\\", config[\\"artifacts\\"][\\"dpo_adapter_path\\"]),\\n    }\\n    report[\\"adapters\\"] = {}\\n    for label, (expected_suffix, raw_path) in adapters.items():\\n        adapter_dir = Path(raw_path)\\n        base_model = _adapter_base_model(adapter_dir)\\n        if base_model is None:\\n            errors.append(f\\"{label}: no readable adapter_config.json under {adapter_dir}\\")\\n        elif not base_model.endswith(expected_suffix):\\n            errors.append(\\n                f\\"{label}: adapter at {adapter_dir} records base model {base_model!r}, \\"\\n                f\\"which does not end with {expected_suffix!r}\\"\\n            )\\n        report[\\"adapters\\"][label] = {\\"path\\": str(adapter_dir), \\"base_model\\": base_model}\\n\\n    corpus_path = Path(config[\\"data\\"][\\"corpus_path\\"])\\n    if not corpus_path.is_file():\\n        errors.append(f\\"corpus_path not found: {corpus_path}\\")\\n        corpus: list[dict[str, Any]] = []\\n    else:\\n        corpus = load_corpus(corpus_path)\\n        chunk_ids = [row.get(\\"chunk_id\\") for row in corpus]\\n        if any(not isinstance(cid, str) or not cid for cid in chunk_ids):\\n            errors.append(\\"corpus has rows with a missing or non-string chunk_id\\")\\n        elif len(set(chunk_ids)) != len(chunk_ids):\\n            errors.append(\\"corpus chunk_ids are not unique\\")\\n        if any(not str(row.get(\\"text\\", \\"\\")).strip() for row in corpus):\\n            errors.append(\\"corpus has rows with empty text\\")\\n    report[\\"corpus_count\\"] = len(corpus)\\n\\n    for name, value in (\\n        (\\"OPENAI_API_KEY\\", OPENAI_API_KEY),\\n        (\\"OPENAI_BASE_URL\\", OPENAI_BASE_URL),\\n        (\\"GEMINI_API_KEY\\", GEMINI_API_KEY),\\n        (\\"GEMINI_ENDPOINT\\", GEMINI_ENDPOINT),\\n    ):\\n        if not value:\\n            errors.append(f\\"{name} is empty; set it in the environment, never in the config\\")\\n\\n    if int(config[\\"retrieval\\"][\\"top_k\\"]) < 1:\\n        errors.append(\\"retrieval.top_k must be at least 1\\")\\n    if int(config[\\"execution\\"][\\"max_workers\\"]) < 1:\\n        errors.append(\\"execution.max_workers must be at least 1\\")\\n    for label, block in (\\n        (\\"generator.retry\\", config[\\"generator\\"][\\"retry\\"]),\\n        (\\"judges.primary.retry\\", config[\\"judges\\"][\\"primary\\"][\\"retry\\"]),\\n        (\\"judges.secondary.retry\\", config[\\"judges\\"][\\"secondary\\"][\\"retry\\"]),\\n    ):\\n        try:\\n            RetryPolicy.from_config(block, label)\\n        except (KeyError, TypeError, ValueError) as exc:\\n            errors.append(f\\"{label}: {exc}\\")\\n\\n    output_dir = Path(config[\\"output_dir\\"])\\n    try:\\n        output_dir.mkdir(parents=True, exist_ok=True)\\n        probe_path = output_dir / \\".write_probe\\"\\n        probe_path.write_text(\\"\\", encoding=\\"utf-8\\")\\n        probe_path.unlink()\\n    except OSError as exc:\\n        errors.append(f\\"output_dir {output_dir} is not writable: {exc}\\")\\n\\n    if errors:\\n        raise ValueError(\\"preflight failed:\\\\n  - \\" + \\"\\\\n  - \\".join(errors))\\n\\n    report[\\"expected_keys\\"] = (\\n        report[\\"n_questions\\"] * len(report[\\"personas\\"]) * len(arms) * len(replicates)\\n    )\\n    report[\\"probes\\"] = probe_models(config) if probe else {}\\n    return report\\n\\n\\ndef print_preflight(report: dict[str, Any]) -> None:\\n    print(f\\"Questions:        {report[\'n_questions\']}\\")\\n    personas = \\", \\".join(f\\"{p[\'id\']} ({p[\'split\']})\\" for p in report[\\"personas\\"])\\n    print(f\\"Personas:         {personas}\\")\\n    print(f\\"Replicates:       {report[\'replicates\']}\\")\\n    print(f\\"Arms:             {\', \'.join(report[\'arms\'])}\\")\\n    for label, info in report[\\"adapters\\"].items():\\n        print(f\\"{label:<17} {info[\'path\']} -> {info[\'base_model\']}\\")\\n    print(f\\"Corpus chunks:    {report[\'corpus_count\']}\\")\\n    for model, ok in report[\\"probes\\"].items():\\n        print(f\\"Probe {model:<20} {\'ok\' if ok else \'EMPTY REPLY\'}\\")\\n    print(f\\"Expected keys:    {report[\'expected_keys\']}\\")\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Prompting and remote calls\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef build_answer_messages(\\n    profile_rendered: str | None, query: str, chunks: list[RetrievedChunk]\\n) -> list[dict[str, str]]:\\n    \\"\\"\\"Build the generator turn.\\n\\n    Not ``rag.prompts.build_mcq_rag_prompt``: that helper takes options as a separate\\n    argument and forces multiple-choice framing, while ``data.questions.load_question``\\n    already renders passages, options, pairs, and items into one string — and several\\n    held-out questions are not multiple choice.\\n\\n    The *original* question is shown here even for the rewriter arms. Rewriting is a\\n    retrieval-side intervention; showing the model a rewritten question would change what\\n    it is asked to answer and make the arms incomparable.\\n    \\"\\"\\"\\n    if chunks:\\n        passages = \\"\\\\n\\\\n\\".join(f\\"[{chunk.rank}] {chunk.text}\\" for chunk in chunks)\\n    else:\\n        passages = \\"(no passages were retrieved)\\"\\n    system = ANSWER_SYSTEM if profile_rendered is None else ANSWER_SYSTEM_PERSONALIZED\\n    prefix = \\"\\" if profile_rendered is None else f\\"Learner profile: {profile_rendered}\\\\n\\\\n\\"\\n    user = f\\"{prefix}Passages:\\\\n{passages}\\\\n\\\\nQuestion:\\\\n{query}\\"\\n    return [{\\"role\\": \\"system\\", \\"content\\": system}, {\\"role\\": \\"user\\", \\"content\\": user}]\\n\\n\\ndef _call_with_retry(call: Callable[[], str], retry: RetryPolicy, label: str) -> tuple[str, int]:\\n    \\"\\"\\"Retry *call* until it returns non-empty text; return ``(text, attempts)``.\\n\\n    An empty completion is a failed attempt, not an answer: judged as-is it would score a\\n    faithful zero and quietly depress whichever arm hit the endpoint on a bad minute.\\n    \\"\\"\\"\\n    delay = retry.initial_backoff_seconds\\n    last_error = \\"no attempt was made\\"\\n    for attempt in range(1, retry.max_attempts + 1):\\n        try:\\n            text = call()\\n            if text and text.strip():\\n                return text, attempt\\n            last_error = \\"empty completion\\"\\n        except Exception as exc:  # transport failures retry like empty replies\\n            last_error = f\\"{type(exc).__name__}: {exc}\\"\\n        if attempt < retry.max_attempts:\\n            time.sleep(delay)\\n            delay = min(delay * retry.backoff_multiplier, retry.max_backoff_seconds)\\n    raise RemoteCallError(\\n        f\\"{label} failed after {retry.max_attempts} attempts; last error: {last_error}\\"\\n    )\\n\\n\\n# --------------------------------------------------------------------------------------\\n# GPU stages\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef _free(*objects: Any) -> None:\\n    import torch\\n\\n    for obj in objects:\\n        del obj\\n    gc.collect()\\n    if torch.cuda.is_available():\\n        torch.cuda.empty_cache()\\n\\n\\ndef _build_index(embedder: Any, texts: list[str]) -> Any:\\n    import faiss\\n\\n    vectors = embedder.encode(texts)\\n    index = faiss.IndexFlatIP(embedder.dim)\\n    index.add(vectors)\\n    if index.ntotal != len(texts):\\n        raise RuntimeError(f\\"index holds {index.ntotal} vectors for {len(texts)} chunks\\")\\n    return index\\n\\n\\ndef _retrieve_for_arm(\\n    embedder: Any,\\n    index: Any,\\n    corpus: list[dict[str, Any]],\\n    arm: dict[str, Any],\\n    cases: list[Case],\\n    queries: list[str],\\n    top_k: int,\\n) -> dict[tuple[str, str], ArmContext]:\\n    \\"\\"\\"Retrieve one arm\'s passages for every case.\\n\\n    Queries are grouped by instruction, because ``encode_query`` applies one instruction\\n    per call and cannot mix them within a batch, and deduplicated within each group. The\\n    dedup is what makes the unprofiled arm cost one encode per question instead of one per\\n    persona: with no profile in the query, four personas submit identical text.\\n    \\"\\"\\"\\n    grouped: dict[str, dict[str, list[int]]] = {}\\n    for position, (case, query) in enumerate(zip(cases, queries, strict=True)):\\n        instruction = case.profile_rendered if arm[\\"profile\\"] else \\"\\"\\n        grouped.setdefault(instruction, {}).setdefault(query, []).append(position)\\n\\n    contexts: dict[tuple[str, str], ArmContext] = {}\\n    for instruction, by_query in grouped.items():\\n        unique_queries = list(by_query)\\n        vectors = embedder.encode_query(unique_queries, instruction=instruction)\\n        scores, indices = index.search(vectors, top_k)\\n        for row, query in enumerate(unique_queries):\\n            chunks = [\\n                RetrievedChunk(\\n                    chunk_id=corpus[int(corpus_index)][\\"chunk_id\\"],\\n                    text=corpus[int(corpus_index)][\\"text\\"],\\n                    score=float(score),\\n                    rank=rank,\\n                )\\n                for rank, (corpus_index, score) in enumerate(\\n                    zip(indices[row], scores[row], strict=True), start=1\\n                )\\n                if int(corpus_index) >= 0\\n            ]\\n            for position in by_query[query]:\\n                case = cases[position]\\n                contexts[case.question_ref, case.persona_id] = ArmContext(\\n                    retrieval_query=query,\\n                    retrieval_instruction=instruction,\\n                    chunks=chunks,\\n                )\\n    return contexts\\n\\n\\ndef _rewrite_all(\\n    config: dict[str, Any], adapter_path: str | None, cases: list[Case], device: str\\n) -> list[str]:\\n    from rag.rewriter import DPORewriter\\n\\n    rewriter_config = config[\\"rewriter\\"]\\n    rewriter = DPORewriter(\\n        model_name=rewriter_config[\\"model_name\\"],\\n        adapter_path=adapter_path,\\n        device=device,\\n        max_seq_length=rewriter_config[\\"max_seq_length\\"],\\n        generation=rewriter_config[\\"generation\\"],\\n    )\\n    batch_size = int(rewriter_config[\\"batch_size\\"])\\n    rewrites: list[str] = []\\n    for start in range(0, len(cases), batch_size):\\n        batch = cases[start : start + batch_size]\\n        rewrites.extend(\\n            rewriter.rewrite_batch(\\n                [case.profile_rendered for case in batch], [case.query for case in batch]\\n            )\\n        )\\n    _free(rewriter)\\n    return rewrites\\n\\n\\ndef run_gpu_stages(\\n    config: dict[str, Any], cases: list[Case], local_rank: int\\n) -> dict[str, dict[tuple[str, str], ArmContext]]:\\n    \\"\\"\\"Stages A-C: every arm\'s retrieval, four model loads, one at a time in memory.\\"\\"\\"\\n    corpus = load_corpus(config[\\"data\\"][\\"corpus_path\\"])\\n    texts = [row[\\"text\\"] for row in corpus]\\n    top_k = int(config[\\"retrieval\\"][\\"top_k\\"])\\n    device = f\\"cuda:{local_rank}\\"\\n    arms_by_name = {arm[\\"name\\"]: arm for arm in config[\\"arms\\"]}\\n    contexts: dict[str, dict[tuple[str, str], ArmContext]] = {}\\n\\n    from rag.embedder import Qwen3Embedder\\n\\n    embedder_config = config[\\"embedder\\"]\\n\\n    def make_embedder(adapter_path: str | None) -> Qwen3Embedder:\\n        return Qwen3Embedder(\\n            model_name=embedder_config[\\"model_name\\"],\\n            device=device,\\n            batch_size=int(embedder_config[\\"batch_size\\"]),\\n            fp16=bool(embedder_config[\\"fp16\\"]),\\n            adapter_path=adapter_path,\\n            max_seq_length=int(embedder_config[\\"max_seq_length\\"]),\\n        )\\n\\n    # Stage A: base embedder, base index.\\n    base_arms = [arm for arm in config[\\"arms\\"] if arm[\\"embedder\\"] == \\"base\\"]\\n    if base_arms:\\n        embedder = make_embedder(None)\\n        index = _build_index(embedder, texts)\\n        for arm in base_arms:\\n            if arm[\\"rewriter\\"] != \\"none\\":\\n                raise ValueError(f\\"arm {arm[\'name\']}: base embedder arms take no rewriter\\")\\n            contexts[arm[\\"name\\"]] = _retrieve_for_arm(\\n                embedder, index, corpus, arm, cases, [case.query for case in cases], top_k\\n            )\\n        _free(embedder, index)\\n\\n    # Stage B: rewriters, before the trained embedder is loaded. Two separate loads because\\n    # DPORewriter exposes no adapter toggle and adding one is out of scope here.\\n    rewrites: dict[str, list[str]] = {}\\n    needed = {arm[\\"rewriter\\"] for arm in config[\\"arms\\"]} - {\\"none\\"}\\n    if \\"base\\" in needed:\\n        rewrites[\\"base\\"] = _rewrite_all(config, None, cases, device)\\n    if \\"dpo\\" in needed:\\n        rewrites[\\"dpo\\"] = _rewrite_all(\\n            config, config[\\"artifacts\\"][\\"dpo_adapter_path\\"], cases, device\\n        )\\n\\n    # Stage C: ROPG embedder. Run B was promoted with anchor mode `both`\\n    # (docs/results/ropg-runs-comparison-v1.md), so both towers are adapted and the served\\n    # index must be built with the adapter, not just the query side.\\n    ropg_arms = [arm for arm in config[\\"arms\\"] if arm[\\"embedder\\"] == \\"ropg\\"]\\n    if ropg_arms:\\n        embedder = make_embedder(config[\\"artifacts\\"][\\"ropg_adapter_path\\"])\\n        index = _build_index(embedder, texts)\\n        for arm in ropg_arms:\\n            kind = arm[\\"rewriter\\"]\\n            queries = [case.query for case in cases] if kind == \\"none\\" else rewrites[kind]\\n            contexts[arm[\\"name\\"]] = _retrieve_for_arm(\\n                embedder, index, corpus, arm, cases, queries, top_k\\n            )\\n        _free(embedder, index)\\n\\n    missing = set(arms_by_name) - set(contexts)\\n    if missing:\\n        raise RuntimeError(f\\"no retrieval was produced for arms {sorted(missing)}\\")\\n    return contexts\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Stage D: remote generation and judging\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef _load_completed(rank_path: Path, digest: str) -> tuple[set[tuple[int, str, str, str]], int]:\\n    \\"\\"\\"Return keys already finished under *digest*, and the count of foreign-config rows.\\"\\"\\"\\n    completed: set[tuple[int, str, str, str]] = set()\\n    stale = 0\\n    if not rank_path.is_file():\\n        return completed, stale\\n    with rank_path.open(encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            line = line.strip()\\n            if not line:\\n                continue\\n            record = json.loads(line)\\n            if record.get(\\"config_sha256\\") != digest:\\n                stale += 1\\n                continue\\n            if record.get(\\"status\\") == \\"ok\\":\\n                completed.add(\\n                    (\\n                        record[\\"replicate\\"],\\n                        record[\\"arm\\"],\\n                        record[\\"question_ref\\"],\\n                        record[\\"persona_id\\"],\\n                    )\\n                )\\n    return completed, stale\\n\\n\\ndef run_remote_stage(\\n    config: dict[str, Any],\\n    cases: list[Case],\\n    contexts: dict[str, dict[tuple[str, str], ArmContext]],\\n    rank_path: Path,\\n    digest: str,\\n    rank: int,\\n) -> int:\\n    \\"\\"\\"Generate and judge every pending key on this rank. Returns the number written.\\"\\"\\"\\n    completed, _stale = _load_completed(rank_path, digest)\\n    case_by_key = {(case.question_ref, case.persona_id): case for case in cases}\\n    arms_by_name = {arm[\\"name\\"]: arm for arm in config[\\"arms\\"]}\\n\\n    pending = [key for key in build_expected_keys(config, cases) if key not in completed]\\n    if not pending:\\n        return 0\\n\\n    from rag.llm import OpenAICompatClient\\n\\n    generator_config = config[\\"generator\\"]\\n    generator = OpenAICompatClient(\\n        base_url=OPENAI_BASE_URL,\\n        api_key=OPENAI_API_KEY,\\n        model=generator_config[\\"model\\"],\\n        temperature=float(generator_config[\\"temperature\\"]),\\n        max_tokens=int(generator_config[\\"max_completion_tokens\\"]),\\n    )\\n    generator_retry = RetryPolicy.from_config(generator_config[\\"retry\\"], \\"generator.retry\\")\\n\\n    primary_config = config[\\"judges\\"][\\"primary\\"]\\n    primary = LunaJudge(\\n        model=primary_config[\\"model\\"],\\n        temperature=float(primary_config[\\"temperature\\"]),\\n        reasoning_effort=primary_config.get(\\"reasoning_effort\\"),\\n        max_completion_tokens=int(primary_config[\\"max_completion_tokens\\"]),\\n        retry=RetryPolicy.from_config(primary_config[\\"retry\\"], \\"judges.primary.retry\\"),\\n    )\\n    secondary_config = config[\\"judges\\"][\\"secondary\\"]\\n    secondary = GeminiJudge(\\n        model=secondary_config[\\"model\\"],\\n        temperature=float(secondary_config[\\"temperature\\"]),\\n        thinking_level=secondary_config.get(\\"thinking_level\\"),\\n        max_output_tokens=int(secondary_config[\\"max_output_tokens\\"]),\\n        retry=RetryPolicy.from_config(secondary_config[\\"retry\\"], \\"judges.secondary.retry\\"),\\n    )\\n\\n    ropg_path = config[\\"artifacts\\"][\\"ropg_adapter_path\\"]\\n    dpo_path = config[\\"artifacts\\"][\\"dpo_adapter_path\\"]\\n    lock = threading.Lock()\\n    handle = rank_path.open(\\"a\\", encoding=\\"utf-8\\")\\n\\n    def evaluate(key: tuple[int, str, str, str]) -> None:\\n        replicate, arm_name, question_ref, persona_id = key\\n        arm = arms_by_name[arm_name]\\n        case = case_by_key[question_ref, persona_id]\\n        context = contexts[arm_name][question_ref, persona_id]\\n        passages = [(chunk.chunk_id, chunk.text) for chunk in context.chunks]\\n\\n        record: dict[str, Any] = {\\n            \\"replicate\\": replicate,\\n            \\"arm\\": arm_name,\\n            \\"question_ref\\": question_ref,\\n            \\"persona_id\\": persona_id,\\n            \\"status\\": \\"ok\\",\\n            \\"failure_stage\\": None,\\n            \\"rank\\": rank,\\n            \\"config_sha256\\": digest,\\n            \\"embedder_model\\": config[\\"embedder\\"][\\"model_name\\"],\\n            \\"embedder_adapter_path\\": ropg_path if arm[\\"embedder\\"] == \\"ropg\\" else None,\\n            \\"rewriter_model\\": config[\\"rewriter\\"][\\"model_name\\"]\\n            if arm[\\"rewriter\\"] != \\"none\\"\\n            else None,\\n            \\"rewriter_adapter_path\\": dpo_path if arm[\\"rewriter\\"] == \\"dpo\\" else None,\\n            \\"generator_model\\": generator_config[\\"model\\"],\\n            \\"primary_judge_model\\": primary_config[\\"model\\"],\\n            \\"secondary_judge_model\\": secondary_config[\\"model\\"],\\n            \\"retrieval_instruction\\": context.retrieval_instruction,\\n            \\"original_query\\": case.query,\\n            \\"rewritten_query\\": (context.retrieval_query if arm[\\"rewriter\\"] != \\"none\\" else None),\\n            \\"hits\\": [chunk.as_dict() for chunk in context.chunks],\\n            \\"answer\\": None,\\n            \\"primary_scores\\": None,\\n            \\"secondary_scores\\": None,\\n            \\"primary_attempts\\": 0,\\n            \\"secondary_attempts\\": 0,\\n            \\"primary_error\\": None,\\n            \\"secondary_error\\": None,\\n            \\"timestamp\\": datetime.now(UTC).isoformat(),\\n        }\\n\\n        messages = build_answer_messages(\\n            case.profile_rendered if arm[\\"profile\\"] else None, case.query, context.chunks\\n        )\\n        try:\\n            answer, _attempts = _call_with_retry(\\n                lambda: generator.chat(messages), generator_retry, generator_config[\\"model\\"]\\n            )\\n        except RemoteCallError as exc:\\n            record[\\"status\\"] = \\"failed\\"\\n            record[\\"failure_stage\\"] = \\"generation\\"\\n            record[\\"primary_error\\"] = str(exc)\\n            _append(handle, lock, record)\\n            return\\n        record[\\"answer\\"] = answer\\n\\n        # The judge always sees the learner profile, including for the unprofiled arm:\\n        # persona_alignment on a naive-RAG answer is exactly the number that rung exists\\n        # to establish.\\n        judge_kwargs = {\\n            \\"profile_rendered\\": case.profile_rendered,\\n            \\"query\\": case.query,\\n            \\"gold_answer\\": case.gold_answer,\\n            \\"gold_explanation\\": case.gold_explanation,\\n            \\"passages\\": passages,\\n            \\"answer\\": answer,\\n        }\\n        try:\\n            scores, attempts = primary.score(**judge_kwargs)\\n        except JudgeError as exc:\\n            record[\\"status\\"] = \\"failed\\"\\n            record[\\"failure_stage\\"] = \\"judge_primary\\"\\n            record[\\"primary_error\\"] = str(exc)\\n            _append(handle, lock, record)\\n            return\\n        record[\\"primary_scores\\"] = scores.as_dict()\\n        record[\\"primary_attempts\\"] = attempts\\n\\n        # A secondary failure is not a row failure: the row still carries a primary score\\n        # and appears everywhere except judge agreement.\\n        try:\\n            scores, attempts = secondary.score(**judge_kwargs)\\n        except JudgeError as exc:\\n            record[\\"secondary_error\\"] = str(exc)\\n        else:\\n            record[\\"secondary_scores\\"] = scores.as_dict()\\n            record[\\"secondary_attempts\\"] = attempts\\n        _append(handle, lock, record)\\n\\n    try:\\n        with ThreadPoolExecutor(max_workers=int(config[\\"execution\\"][\\"max_workers\\"])) as pool:\\n            list(pool.map(evaluate, pending))\\n    finally:\\n        handle.close()\\n    return len(pending)\\n\\n\\ndef _append(handle: Any, lock: threading.Lock, record: dict[str, Any]) -> None:\\n    line = json.dumps(record, ensure_ascii=False)\\n    with lock:\\n        handle.write(line + \\"\\\\n\\")\\n        handle.flush()\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Merge and reporting\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef collect_records(output_dir: Path, world_size: int, digest: str) -> tuple[list[dict], int]:\\n    records: list[dict[str, Any]] = []\\n    stale = 0\\n    for rank in range(world_size):\\n        path = output_dir / f\\"rank{rank}.jsonl\\"\\n        if not path.is_file():\\n            continue\\n        with path.open(encoding=\\"utf-8\\") as handle:\\n            for line in handle:\\n                line = line.strip()\\n                if not line:\\n                    continue\\n                record = json.loads(line)\\n                if record.get(\\"config_sha256\\") != digest:\\n                    stale += 1\\n                    continue\\n                records.append(record)\\n    return records, stale\\n\\n\\ndef resolve_records(\\n    records: list[dict[str, Any]], expected: list[tuple[int, str, str, str]]\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"One record per expected key: the success if there is one, else the last failure.\\"\\"\\"\\n    by_key: dict[tuple[int, str, str, str], list[dict[str, Any]]] = {}\\n    for record in records:\\n        key = (record[\\"replicate\\"], record[\\"arm\\"], record[\\"question_ref\\"], record[\\"persona_id\\"])\\n        by_key.setdefault(key, []).append(record)\\n\\n    resolved: list[dict[str, Any]] = []\\n    duplicated: list[tuple[int, str, str, str]] = []\\n    missing: list[tuple[int, str, str, str]] = []\\n    for key in expected:\\n        found = by_key.get(key, [])\\n        successes = [record for record in found if record.get(\\"status\\") == \\"ok\\"]\\n        if len(successes) > 1:\\n            duplicated.append(key)\\n        elif successes:\\n            resolved.append(successes[0])\\n        elif found:\\n            resolved.append(found[-1])\\n        else:\\n            missing.append(key)\\n    if duplicated:\\n        raise RuntimeError(f\\"{len(duplicated)} keys have more than one success: {duplicated[:5]}\\")\\n    if missing:\\n        raise RuntimeError(f\\"{len(missing)} expected keys have no record at all: {missing[:5]}\\")\\n\\n    unexpected = set(by_key) - set(expected)\\n    if unexpected:\\n        raise RuntimeError(\\n            f\\"{len(unexpected)} records do not belong to this run: {sorted(unexpected)[:5]}\\"\\n        )\\n    return resolved\\n\\n\\ndef _scores(record: dict[str, Any], role: str) -> dict[str, int] | None:\\n    return record.get(f\\"{role}_scores\\")\\n\\n\\ndef _persona_groups(personas: list[str]) -> list[tuple[str, str, set[str]]]:\\n    \\"\\"\\"``(persona_label, persona_split_label, member ids)`` for every summary grouping.\\"\\"\\"\\n    groups = [(p, PERSONAS[p].split, {p}) for p in personas]\\n    for split in (\\"train\\", \\"test\\"):\\n        members = {p for p in personas if PERSONAS[p].split == split}\\n        if members:\\n            groups.append((f\\"all_{split}\\", split, members))\\n    groups.append((\\"all\\", \\"all\\", set(personas)))\\n    return groups\\n\\n\\ndef write_summary(path: Path, records: list[dict[str, Any]], config: dict[str, Any]) -> None:\\n    personas = list(config[\\"personas\\"])\\n    replicate_groups: list[tuple[Any, set[int]]] = [(r, {r}) for r in config[\\"replicates\\"]]\\n    replicate_groups.append((\\"all\\", set(config[\\"replicates\\"])))\\n\\n    with path.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.writer(handle)\\n        writer.writerow(\\n            [\\n                \\"arm\\",\\n                \\"persona\\",\\n                \\"persona_split\\",\\n                \\"replicate\\",\\n                \\"judge\\",\\n                \\"metric\\",\\n                \\"mean\\",\\n                \\"std\\",\\n                \\"n_valid\\",\\n                \\"n_failed\\",\\n            ]\\n        )\\n        for arm in ARM_NAMES:\\n            for persona_label, split_label, members in _persona_groups(personas):\\n                for replicate_label, replicate_members in replicate_groups:\\n                    # Only the pooled-replicate row is emitted for the pooled persona\\n                    # groupings: a per-replicate, per-split cell answers no question the\\n                    # other rows do not, and multiplies the file for nothing.\\n                    if persona_label.startswith(\\"all\\") and replicate_label != \\"all\\":\\n                        continue\\n                    rows = [\\n                        record\\n                        for record in records\\n                        if record[\\"arm\\"] == arm\\n                        and record[\\"persona_id\\"] in members\\n                        and record[\\"replicate\\"] in replicate_members\\n                    ]\\n                    if not rows:\\n                        continue\\n                    for role in JUDGE_ROLES:\\n                        for metric in SCORE_FIELDS:\\n                            values = [\\n                                _scores(record, role)[metric]\\n                                for record in rows\\n                                if _scores(record, role) is not None\\n                            ]\\n                            n_valid = len(values)\\n                            array = np.asarray(values, dtype=float)\\n                            mean = float(array.mean()) if n_valid else \\"\\"\\n                            std = float(array.std(ddof=1)) if n_valid > 1 else 0.0\\n                            writer.writerow(\\n                                [\\n                                    arm,\\n                                    persona_label,\\n                                    split_label,\\n                                    replicate_label,\\n                                    role,\\n                                    metric,\\n                                    mean,\\n                                    std,\\n                                    n_valid,\\n                                    len(rows) - n_valid,\\n                                ]\\n                            )\\n\\n\\ndef write_paired_deltas(path: Path, records: list[dict[str, Any]], config: dict[str, Any]) -> None:\\n    \\"\\"\\"Adjacent-rung paired deltas, Holm-corrected across the whole reported family.\\n\\n    Pairing is what buys the power here: the same question and persona is scored under both\\n    arms, so per-question difficulty — the dominant variance term — cancels in the\\n    difference. Holm runs once over every row in the file rather than per comparison,\\n    because the family a reader sees is the file.\\n    \\"\\"\\"\\n    personas = list(config[\\"personas\\"])\\n    persona_groups = [(p, PERSONAS[p].split, {p}) for p in personas]\\n    persona_groups.append((\\"all\\", \\"all\\", set(personas)))\\n    n_boot = int(config[\\"execution\\"][\\"bootstrap_samples\\"])\\n    seed = int(config[\\"execution\\"][\\"bootstrap_seed\\"])\\n\\n    indexed: dict[tuple[str, str, str, int], dict[str, Any]] = {\\n        (r[\\"arm\\"], r[\\"question_ref\\"], r[\\"persona_id\\"], r[\\"replicate\\"]): r for r in records\\n    }\\n    rows: list[list[Any]] = []\\n    for comparison, before_arm, after_arm in COMPARISONS:\\n        for persona_label, split_label, members in persona_groups:\\n            for role in JUDGE_ROLES:\\n                for metric in SCORE_FIELDS:\\n                    before_values: list[float] = []\\n                    after_values: list[float] = []\\n                    for question_ref, persona_id, replicate in sorted(\\n                        {\\n                            (r[\\"question_ref\\"], r[\\"persona_id\\"], r[\\"replicate\\"])\\n                            for r in records\\n                            if r[\\"persona_id\\"] in members\\n                        }\\n                    ):\\n                        before = indexed.get((before_arm, question_ref, persona_id, replicate))\\n                        after = indexed.get((after_arm, question_ref, persona_id, replicate))\\n                        if before is None or after is None:\\n                            continue\\n                        before_scores = _scores(before, role)\\n                        after_scores = _scores(after, role)\\n                        if before_scores is None or after_scores is None:\\n                            continue\\n                        before_values.append(before_scores[metric])\\n                        after_values.append(after_scores[metric])\\n                    if not before_values:\\n                        continue\\n                    stats = paired_stats(\\n                        np.asarray(before_values, dtype=float),\\n                        np.asarray(after_values, dtype=float),\\n                        n_boot=n_boot,\\n                        rng=np.random.default_rng(seed),\\n                    )\\n                    rows.append(\\n                        [\\n                            comparison,\\n                            persona_label,\\n                            split_label,\\n                            role,\\n                            metric,\\n                            stats[\\"before\\"],\\n                            stats[\\"after\\"],\\n                            stats[\\"delta\\"],\\n                            stats[\\"lo\\"],\\n                            stats[\\"hi\\"],\\n                            stats[\\"p\\"],\\n                            None,\\n                            stats[\\"n\\"],\\n                        ]\\n                    )\\n\\n    adjusted = holm([row[10] for row in rows]) if rows else []\\n    for row, p_holm in zip(rows, adjusted, strict=True):\\n        row[11] = p_holm\\n\\n    with path.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.writer(handle)\\n        writer.writerow(\\n            [\\n                \\"comparison\\",\\n                \\"persona\\",\\n                \\"persona_split\\",\\n                \\"judge\\",\\n                \\"metric\\",\\n                \\"before\\",\\n                \\"after\\",\\n                \\"delta\\",\\n                \\"ci_lo\\",\\n                \\"ci_hi\\",\\n                \\"p\\",\\n                \\"p_holm\\",\\n                \\"n\\",\\n            ]\\n        )\\n        writer.writerows(rows)\\n\\n\\ndef write_judge_agreement(path: Path, records: list[dict[str, Any]]) -> None:\\n    \\"\\"\\"How far the two judges agree, pooled and per arm.\\n\\n    Per arm is not optional: disagreement concentrated in one arm is what reward hacking\\n    against the primary judge looks like from the outside.\\n    \\"\\"\\"\\n    groups: list[tuple[str, list[dict[str, Any]]]] = [(\\"all\\", records)]\\n    groups.extend(\\n        (arm, [record for record in records if record[\\"arm\\"] == arm]) for arm in ARM_NAMES\\n    )\\n    with path.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.writer(handle)\\n        writer.writerow(\\n            [\\"arm\\", \\"persona\\", \\"metric\\", \\"n\\", \\"exact_agreement_rate\\", \\"mean_abs_diff\\", \\"pearson_r\\"]\\n        )\\n        for arm_label, rows in groups:\\n            both = [\\n                record\\n                for record in rows\\n                if _scores(record, \\"primary\\") is not None\\n                and _scores(record, \\"secondary\\") is not None\\n            ]\\n            for metric in SCORE_FIELDS:\\n                if not both:\\n                    writer.writerow([arm_label, \\"all\\", metric, 0, \\"\\", \\"\\", \\"\\"])\\n                    continue\\n                a = np.asarray([_scores(r, \\"primary\\")[metric] for r in both], dtype=float)\\n                b = np.asarray([_scores(r, \\"secondary\\")[metric] for r in both], dtype=float)\\n                exact = float((a == b).mean())\\n                mad = float(np.abs(a - b).mean())\\n                # Degenerate on a constant vector, which happens whenever a metric\\n                # saturates. numpy warns and returns nan; report the nan rather than a 0.\\n                if a.std() == 0 or b.std() == 0:\\n                    corr = float(\\"nan\\")\\n                else:\\n                    corr = float(np.corrcoef(a, b)[0, 1])\\n                writer.writerow([arm_label, \\"all\\", metric, len(both), exact, mad, corr])\\n\\n\\ndef _package_version(name: str) -> str | None:\\n    import importlib.metadata\\n\\n    try:\\n        return importlib.metadata.version(name)\\n    except importlib.metadata.PackageNotFoundError:\\n        return None\\n\\n\\ndef write_manifest(\\n    path: Path,\\n    config: dict[str, Any],\\n    digest: str,\\n    records: list[dict[str, Any]],\\n    expected: list[tuple[int, str, str, str]],\\n    world_size: int,\\n    stale_records: int,\\n) -> None:\\n    failures: dict[str, int] = {}\\n    for record in records:\\n        if record[\\"status\\"] != \\"ok\\":\\n            stage = record[\\"failure_stage\\"] or \\"unknown\\"\\n            failures[stage] = failures.get(stage, 0) + 1\\n    secondary_missing = sum(\\n        1 for record in records if record[\\"status\\"] == \\"ok\\" and record[\\"secondary_scores\\"] is None\\n    )\\n    per_rank: dict[str, int] = {}\\n    for record in records:\\n        key = str(record[\\"rank\\"])\\n        per_rank[key] = per_rank.get(key, 0) + 1\\n\\n    manifest = {\\n        \\"config\\": config,\\n        \\"config_sha256\\": digest,\\n        \\"corpus\\": {\\n            \\"path\\": config[\\"data\\"][\\"corpus_path\\"],\\n            \\"sha256\\": sha256_file(config[\\"data\\"][\\"corpus_path\\"]),\\n            \\"count\\": len(load_corpus(config[\\"data\\"][\\"corpus_path\\"])),\\n        },\\n        \\"test_qids\\": {\\n            \\"path\\": config[\\"data\\"][\\"test_qids_path\\"],\\n            \\"sha256\\": sha256_file(config[\\"data\\"][\\"test_qids_path\\"]),\\n            \\"count\\": len(read_test_qids(config[\\"data\\"][\\"test_qids_path\\"])),\\n        },\\n        \\"personas\\": [\\n            {\\"id\\": p, \\"split\\": PERSONAS[p].split, \\"rendered\\": PERSONAS[p].rendered}\\n            for p in config[\\"personas\\"]\\n        ],\\n        \\"arms\\": config[\\"arms\\"],\\n        \\"replicates\\": config[\\"replicates\\"],\\n        \\"adapters\\": {\\n            label: {\\n                \\"path\\": str(Path(path_value)),\\n                \\"base_model\\": _adapter_base_model(Path(path_value)),\\n            }\\n            for label, path_value in (\\n                (\\"ropg\\", config[\\"artifacts\\"][\\"ropg_adapter_path\\"]),\\n                (\\"dpo\\", config[\\"artifacts\\"][\\"dpo_adapter_path\\"]),\\n            )\\n        },\\n        \\"models\\": {\\n            \\"generator\\": {\\n                \\"model\\": config[\\"generator\\"][\\"model\\"],\\n                \\"temperature\\": config[\\"generator\\"][\\"temperature\\"],\\n                \\"max_completion_tokens\\": config[\\"generator\\"][\\"max_completion_tokens\\"],\\n            },\\n            \\"judge_primary\\": dict(config[\\"judges\\"][\\"primary\\"]),\\n            \\"judge_secondary\\": dict(config[\\"judges\\"][\\"secondary\\"]),\\n        },\\n        \\"world_size\\": world_size,\\n        \\"records_per_rank\\": per_rank,\\n        \\"expected_keys\\": len(expected),\\n        \\"merged_keys\\": len(records),\\n        \\"successful_keys\\": sum(1 for record in records if record[\\"status\\"] == \\"ok\\"),\\n        \\"failures_by_stage\\": failures,\\n        \\"secondary_judge_missing\\": secondary_missing,\\n        \\"stale_records\\": stale_records,\\n        \\"versions\\": {\\n            name: _package_version(name)\\n            for name in (\\n                \\"torch\\",\\n                \\"transformers\\",\\n                \\"peft\\",\\n                \\"sentence-transformers\\",\\n                \\"faiss-gpu\\",\\n                \\"faiss-cpu\\",\\n                \\"openai\\",\\n                \\"google-genai\\",\\n            )\\n        },\\n    }\\n    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n\\n\\ndef merge(\\n    config: dict[str, Any],\\n    cases: list[Case],\\n    digest: str,\\n    world_size: int,\\n) -> dict[str, Any]:\\n    output_dir = Path(config[\\"output_dir\\"])\\n    expected = build_expected_keys(config, cases)\\n    records, stale = collect_records(output_dir, world_size, digest)\\n    resolved = resolve_records(records, expected)\\n    resolved.sort(key=lambda r: (r[\\"replicate\\"], r[\\"arm\\"], r[\\"question_ref\\"], r[\\"persona_id\\"]))\\n\\n    results_path = output_dir / \\"results.jsonl\\"\\n    with results_path.open(\\"w\\", encoding=\\"utf-8\\") as handle:\\n        for record in resolved:\\n            handle.write(json.dumps(record, ensure_ascii=False) + \\"\\\\n\\")\\n\\n    successful = [record for record in resolved if record[\\"status\\"] == \\"ok\\"]\\n    write_summary(output_dir / \\"summary.csv\\", successful, config)\\n    write_paired_deltas(output_dir / \\"paired_deltas.csv\\", successful, config)\\n    write_judge_agreement(output_dir / \\"judge_agreement.csv\\", successful)\\n    write_manifest(\\n        output_dir / \\"run_manifest.json\\", config, digest, resolved, expected, world_size, stale\\n    )\\n    return {\\n        \\"expected\\": len(expected),\\n        \\"merged\\": len(resolved),\\n        \\"successful\\": len(successful),\\n        \\"stale\\": stale,\\n    }\\n\\n\\n# --------------------------------------------------------------------------------------\\n# Entry point\\n# --------------------------------------------------------------------------------------\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--config\\", required=True)\\n    parser.add_argument(\\"--preflight-only\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--limit\\", type=int, default=None)\\n    args = parser.parse_args(argv)\\n\\n    config = load_config(args.config)\\n    digest = config_digest(config)\\n\\n    rank = int(os.environ.get(\\"RANK\\", \\"0\\"))\\n    local_rank = int(os.environ.get(\\"LOCAL_RANK\\", \\"0\\"))\\n    world_size = int(os.environ.get(\\"WORLD_SIZE\\", \\"1\\"))\\n\\n    report = validate(config)\\n    if args.preflight_only:\\n        if rank == 0:\\n            print_preflight(report)\\n        return 0\\n\\n    cases = build_cases(config, args.limit)\\n    shard = cases[rank::world_size]\\n    output_dir = Path(config[\\"output_dir\\"])\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    rank_path = output_dir / f\\"rank{rank}.jsonl\\"\\n\\n    if world_size > 1:\\n        import torch\\n        import torch.distributed as dist\\n\\n        torch.cuda.set_device(local_rank)\\n        # gloo, not NCCL: this job is inference-only and its one collective is a barrier,\\n        # so no GPU tensor is ever communicated.\\n        dist.init_process_group(\\n            backend=\\"gloo\\",\\n            init_method=\\"env://\\",\\n            rank=rank,\\n            world_size=world_size,\\n            timeout=timedelta(minutes=10),\\n        )\\n\\n    print(f\\"[rank {rank}] {len(shard)} cases, config {digest[:12]}\\", flush=True)\\n    contexts = run_gpu_stages(config, shard, local_rank)\\n    written = run_remote_stage(config, shard, contexts, rank_path, digest, rank)\\n    print(f\\"[rank {rank}] wrote {written} records\\", flush=True)\\n\\n    if world_size > 1:\\n        import torch.distributed as dist\\n\\n        dist.barrier()\\n\\n    if rank == 0:\\n        stats = merge(config, cases, digest, world_size)\\n        print(\\n            f\\"Merged {stats[\'merged\']}/{stats[\'expected\']} keys \\"\\n            f\\"({stats[\'successful\']} successful, {stats[\'stale\']} stale) -> {output_dir}\\"\\n        )\\n\\n    if world_size > 1:\\n        import torch.distributed as dist\\n\\n        dist.barrier()\\n        dist.destroy_process_group()\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "benchmarks/judge.py": "\\"\\"\\"Two independent LLM judges scoring one shared rubric.\\n\\nEvery evaluated row is scored twice: once by ``gpt-5.6-luna`` (primary) and once by\\n``gemini-3.7-flash`` (secondary). The redundancy is not belt-and-braces. Luna labelled the\\nDPO preference pairs the rewriter arm was trained on and was the ROPG teacher, so a\\nLuna-only number for those arms grades a student against its own teacher. The secondary\\njudge is independent of both, and ``judge_agreement.csv`` is what makes the pair reportable.\\n\\nBoth judges receive byte-identical system text and byte-identical user text. The only\\ndifferences are the transport (OpenAI-compatible chat versus Google\'s native GenAI route)\\nand the sampling knobs each endpoint exposes.\\n\\nA judge that cannot produce a parseable, in-range score raises :class:`JudgeError`. It never\\nreturns a sentinel, a zero, or a -1: a failure that looks like a score silently drags every\\narm\'s mean toward the failure rate of its endpoint rather than the quality of its answers.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom typing import TYPE_CHECKING, Any\\n\\nfrom data.settings import GEMINI_API_KEY, GEMINI_ENDPOINT, OPENAI_API_KEY, OPENAI_BASE_URL\\n\\nif TYPE_CHECKING:\\n    from collections.abc import Callable\\n\\n#: Scored 0..4.\\nINTEGER_FIELDS = (\\"context_utility\\", \\"faithfulness\\", \\"persona_alignment\\", \\"pedagogical_quality\\")\\n#: Scored 0 or 1.\\nBINARY_FIELDS = (\\"answer_correctness\\",)\\nSCORE_FIELDS = (\\n    \\"context_utility\\",\\n    \\"answer_correctness\\",\\n    \\"faithfulness\\",\\n    \\"persona_alignment\\",\\n    \\"pedagogical_quality\\",\\n)\\n\\nJUDGE_SYSTEM = (\\n    \\"You are an impartial evaluator for a Persian educational RAG system. You receive a \\"\\n    \\"learner profile, a question, the gold answer and its gold explanation, the passages \\"\\n    \\"that were retrieved for this learner, and the response a system produced.\\\\n\\\\n\\"\\n    \\"Score five dimensions.\\\\n\\\\n\\"\\n    \\"context_utility (0-4): how relevant and sufficient the retrieved passages are for \\"\\n    \\"answering this learner\'s question. Judge the passages, not the response.\\\\n\\"\\n    \\"faithfulness (0-4): whether every claim in the response is supported by the retrieved \\"\\n    \\"passages and none contradicts them. A fluent response that goes beyond or against the \\"\\n    \\"passages scores low however correct it sounds.\\\\n\\"\\n    \\"persona_alignment (0-4): how well the depth, vocabulary, and style of the response fit \\"\\n    \\"the stated learner.\\\\n\\"\\n    \\"pedagogical_quality (0-4): whether the response explains the correct answer in a way \\"\\n    \\"that improves this learner\'s understanding.\\\\n\\\\n\\"\\n    \\"Anchors for those four: 0 = failed, 1 = poor, 2 = partial, 3 = good, 4 = fully \\"\\n    \\"satisfied.\\\\n\\\\n\\"\\n    \\"answer_correctness (0 or 1): 1 when the response agrees with the gold answer, 0 when it \\"\\n    \\"does not or leaves it unclear. The gold answer and gold explanation are reference \\"\\n    \\"material for you only; the system that produced the response never saw them.\\\\n\\\\n\\"\\n    \\"Do not award a holistic or overall score. Reply with strict JSON and nothing else: no \\"\\n    \\"markdown, no code fence, no commentary. Exactly these five keys, each an integer:\\\\n\\"\\n    \'{\\"context_utility\\": <int>, \\"answer_correctness\\": <int>, \\"faithfulness\\": <int>, \'\\n    \'\\"persona_alignment\\": <int>, \\"pedagogical_quality\\": <int>}\'\\n)\\n\\n# Pins the reply shape server-side for the Gemini route, mirroring\\n# benchmarks/compare_dpo_rewriters.py. `parse_judge_scores` still runs on the result: a\\n# schema fixes the keys and their types, not their ranges.\\n_GEMINI_SCHEMA = {\\n    \\"type\\": \\"OBJECT\\",\\n    \\"properties\\": {field: {\\"type\\": \\"INTEGER\\"} for field in SCORE_FIELDS},\\n    \\"required\\": list(SCORE_FIELDS),\\n    \\"property_ordering\\": list(SCORE_FIELDS),\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass JudgeScores:\\n    \\"\\"\\"One judge\'s verdict on one response.\\"\\"\\"\\n\\n    context_utility: int\\n    answer_correctness: int\\n    faithfulness: int\\n    persona_alignment: int\\n    pedagogical_quality: int\\n\\n    def as_dict(self) -> dict[str, int]:\\n        return asdict(self)\\n\\n\\n@dataclass(frozen=True)\\nclass RetryPolicy:\\n    \\"\\"\\"Bounded exponential backoff for one remote role.\\n\\n    Redefined here rather than imported from ``benchmarks/compare_dpo_rewriters.py``, which\\n    carries the identical shape: importing that module would pull ``rl.dpo_train`` and\\n    ``rag.rewriter`` — and through them unsloth — into a job that only needs a dataclass.\\n    \\"\\"\\"\\n\\n    max_attempts: int\\n    initial_backoff_seconds: float\\n    backoff_multiplier: float\\n    max_backoff_seconds: float\\n\\n    @classmethod\\n    def from_config(cls, config: dict[str, Any], label: str = \\"retry\\") -> RetryPolicy:\\n        policy = cls(\\n            max_attempts=int(config[\\"max_attempts\\"]),\\n            initial_backoff_seconds=float(config[\\"initial_backoff_seconds\\"]),\\n            backoff_multiplier=float(config[\\"backoff_multiplier\\"]),\\n            max_backoff_seconds=float(config[\\"max_backoff_seconds\\"]),\\n        )\\n        if policy.max_attempts < 1:\\n            raise ValueError(f\\"{label}.max_attempts must be at least 1\\")\\n        if policy.initial_backoff_seconds < 0 or policy.backoff_multiplier < 1:\\n            raise ValueError(f\\"{label} backoff values are invalid\\")\\n        if policy.max_backoff_seconds < policy.initial_backoff_seconds:\\n            raise ValueError(f\\"{label} maximum must not be below its initial delay\\")\\n        return policy\\n\\n\\nclass JudgeError(RuntimeError):\\n    \\"\\"\\"A judge exhausted its retries without returning a parseable, in-range score.\\"\\"\\"\\n\\n\\ndef build_judge_user_message(\\n    profile_rendered: str,\\n    query: str,\\n    gold_answer: str,\\n    gold_explanation: str,\\n    passages: list[tuple[str, str]],\\n    answer: str,\\n) -> str:\\n    \\"\\"\\"Render the judge\'s user turn.\\n\\n    Passages arrive as ``(chunk_id, text)`` and are numbered from 1 in retrieval order, so\\n    the numbers match the citations the generator was told to use. Nothing is truncated:\\n    cutting a Persian passage at a fixed character count severs it mid-sentence and makes\\n    ``faithfulness`` unjudgeable for exactly the rows where grounding is hardest.\\n    \\"\\"\\"\\n    if passages:\\n        passages_block = \\"\\\\n\\\\n\\".join(\\n            f\\"[{index}] {text}\\" for index, (_chunk_id, text) in enumerate(passages, start=1)\\n        )\\n    else:\\n        passages_block = \\"(no passages were retrieved)\\"\\n    return (\\n        f\\"Learner profile:\\\\n{profile_rendered}\\\\n\\\\n\\"\\n        f\\"Question:\\\\n{query}\\\\n\\\\n\\"\\n        f\\"Gold answer:\\\\n{gold_answer}\\\\n\\\\n\\"\\n        f\\"Gold explanation:\\\\n{gold_explanation}\\\\n\\\\n\\"\\n        f\\"Retrieved passages:\\\\n{passages_block}\\\\n\\\\n\\"\\n        f\\"System response:\\\\n{answer}\\"\\n    )\\n\\n\\ndef parse_judge_scores(raw: str) -> JudgeScores:\\n    \\"\\"\\"Parse a judge reply into :class:`JudgeScores`, or raise ``ValueError``.\\n\\n    Strict on purpose, and deliberately without fence stripping. A judge that wraps its\\n    JSON in markdown is not following the rubric prompt, and quietly repairing its output\\n    here would hide that drift behind scores nobody re-reads.\\n    \\"\\"\\"\\n    if not raw or not raw.strip():\\n        raise ValueError(\\"judge returned an empty reply\\")\\n    try:\\n        payload = json.loads(raw)\\n    except json.JSONDecodeError as exc:\\n        raise ValueError(f\\"judge reply is not valid JSON: {exc}\\") from exc\\n    if not isinstance(payload, dict):\\n        raise ValueError(f\\"judge reply is {type(payload).__name__}, expected an object\\")\\n\\n    expected = set(SCORE_FIELDS)\\n    seen = set(payload)\\n    if seen != expected:\\n        missing = sorted(expected - seen)\\n        extra = sorted(seen - expected)\\n        raise ValueError(f\\"judge reply keys wrong: missing={missing} extra={extra}\\")\\n\\n    values: dict[str, int] = {}\\n    for field in SCORE_FIELDS:\\n        value = payload[field]\\n        # bool is an int subclass, and `true` for answer_correctness would otherwise pass\\n        # every range check below and silently score as 1.\\n        if isinstance(value, bool) or not isinstance(value, int):\\n            raise ValueError(f\\"judge reply {field}={value!r} is not an integer\\")\\n        upper = 1 if field in BINARY_FIELDS else 4\\n        if not 0 <= value <= upper:\\n            raise ValueError(f\\"judge reply {field}={value!r} is outside 0..{upper}\\")\\n        values[field] = value\\n    return JudgeScores(**values)\\n\\n\\ndef _score_with_retry(\\n    call: Callable[[], str], retry: RetryPolicy, judge: str\\n) -> tuple[JudgeScores, int]:\\n    \\"\\"\\"Call *call* until it yields a parseable score, then return ``(scores, attempts)``.\\"\\"\\"\\n    delay = retry.initial_backoff_seconds\\n    last_error = \\"no attempt was made\\"\\n    for attempt in range(1, retry.max_attempts + 1):\\n        try:\\n            return parse_judge_scores(call()), attempt\\n        except Exception as exc:  # transport and parse failures retry alike\\n            last_error = f\\"{type(exc).__name__}: {exc}\\"\\n            if attempt < retry.max_attempts:\\n                time.sleep(delay)\\n                delay = min(delay * retry.backoff_multiplier, retry.max_backoff_seconds)\\n    raise JudgeError(\\n        f\\"{judge} failed after {retry.max_attempts} attempts; last error: {last_error}\\"\\n    )\\n\\n\\nclass LunaJudge:\\n    \\"\\"\\"Primary judge over the OpenAI-compatible route.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model: str,\\n        temperature: float,\\n        reasoning_effort: str | None,\\n        max_completion_tokens: int,\\n        retry: RetryPolicy,\\n    ) -> None:\\n        from rag.llm import OpenAICompatClient\\n\\n        self.model = model\\n        self.retry = retry\\n        # OpenAICompatClient\'s `max_tokens` is sent as `max_completion_tokens` (src/rag/llm.py).\\n        self.client = OpenAICompatClient(\\n            base_url=OPENAI_BASE_URL,\\n            api_key=OPENAI_API_KEY,\\n            model=model,\\n            temperature=temperature,\\n            max_tokens=max_completion_tokens,\\n            reasoning_effort=reasoning_effort,\\n        )\\n\\n    def score(\\n        self,\\n        *,\\n        profile_rendered: str,\\n        query: str,\\n        gold_answer: str,\\n        gold_explanation: str,\\n        passages: list[tuple[str, str]],\\n        answer: str,\\n    ) -> tuple[JudgeScores, int]:\\n        user = build_judge_user_message(\\n            profile_rendered, query, gold_answer, gold_explanation, passages, answer\\n        )\\n        messages = [{\\"role\\": \\"system\\", \\"content\\": JUDGE_SYSTEM}, {\\"role\\": \\"user\\", \\"content\\": user}]\\n        return _score_with_retry(lambda: self.client.chat(messages), self.retry, self.model)\\n\\n\\nclass GeminiJudge:\\n    \\"\\"\\"Secondary judge over Metis\'s native Google GenAI route.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model: str,\\n        temperature: float,\\n        thinking_level: str | None,\\n        max_output_tokens: int,\\n        retry: RetryPolicy,\\n    ) -> None:\\n        from rag.llm import GeminiClient\\n\\n        self.model = model\\n        self.retry = retry\\n        self.client = GeminiClient(\\n            base_url=GEMINI_ENDPOINT,\\n            api_key=GEMINI_API_KEY,\\n            model=model,\\n            system_instruction=JUDGE_SYSTEM,\\n            temperature=temperature,\\n            max_output_tokens=max_output_tokens,\\n            thinking_level=thinking_level,\\n            response_schema=_GEMINI_SCHEMA,\\n        )\\n\\n    def score(\\n        self,\\n        *,\\n        profile_rendered: str,\\n        query: str,\\n        gold_answer: str,\\n        gold_explanation: str,\\n        passages: list[tuple[str, str]],\\n        answer: str,\\n    ) -> tuple[JudgeScores, int]:\\n        user = build_judge_user_message(\\n            profile_rendered, query, gold_answer, gold_explanation, passages, answer\\n        )\\n        return _score_with_retry(lambda: self.client.generate(user), self.retry, self.model)\\n", "src/data/questions.py": "\\"\\"\\"Render a question\'s complete retrieval context — one definition, shared by every stage.\\n\\nBoth data-generation stages and the models they feed must agree on the exact string that\\nrepresents a question. The retriever LoRA was distilled on the form produced here\\n(``Passage:`` / ``Question:`` / ``Options:`` / ``Pairs:`` / ``Items:`` sections), so a\\ncaller that reconstructs a query from ``stem`` alone silently queries the encoder with a\\nform it never saw in training.\\n\\nGold fields (``answer``, ``explanation``) travel beside the query in ``QuestionContext``\\nand are deliberately *not* part of it: they are judge-only reference material, and\\nfolding them into a retrieval query would leak the answer into the retriever\'s input.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom dataclasses import dataclass\\nfrom typing import TYPE_CHECKING\\n\\nif TYPE_CHECKING:\\n    from pathlib import Path\\n\\n\\n@dataclass(frozen=True)\\nclass QuestionContext:\\n    \\"\\"\\"A question\'s retrieval query plus its judge-only gold fields.\\"\\"\\"\\n\\n    query: str\\n    answer: object | None = None\\n    explanation: str | None = None\\n\\n\\ndef render_question_value(value: object) -> str:\\n    \\"\\"\\"Render an extracted value without changing string content.\\"\\"\\"\\n    if isinstance(value, str):\\n        return value\\n    return json.dumps(value, ensure_ascii=False, sort_keys=True)\\n\\n\\ndef render_numbered_section(label: str, values: object) -> str | None:\\n    if values is None:\\n        return None\\n    entries = values if isinstance(values, list) else [values]\\n    if not entries:\\n        return None\\n    rendered = \\"\\\\n\\".join(\\n        f\\"{index}. {render_question_value(value)}\\" for index, value in enumerate(entries, start=1)\\n    )\\n    return f\\"{label}:\\\\n{rendered}\\"\\n\\n\\ndef load_question(exam_stem: str, qid: str, questions_dir: Path) -> QuestionContext:\\n    \\"\\"\\"Load and render the complete retrieval context for *qid*.\\"\\"\\"\\n    qfile = questions_dir / f\\"{exam_stem}.json\\"\\n    if not qfile.exists():\\n        raise FileNotFoundError(f\\"Question file not found: {qfile}\\")\\n    data = json.loads(qfile.read_text(encoding=\\"utf-8\\"))\\n    for q in data.get(\\"questions\\", []):\\n        if q[\\"id\\"] == qid:\\n            sections: list[str] = []\\n            group_id = q.get(\\"group_id\\")\\n            if group_id is not None:\\n                passages = data.get(\\"passages\\", data.get(\\"passage\\", []))\\n                matching_passage: object | None = None\\n\\n                if isinstance(passages, dict):\\n                    if passages.get(\\"id\\") == group_id or passages.get(\\"group_id\\") == group_id:\\n                        matching_passage = passages\\n                    else:\\n                        matching_passage = passages.get(group_id)\\n                        if matching_passage is None:\\n                            matching_passage = passages.get(str(group_id))\\n                elif isinstance(passages, list):\\n                    for passage in passages:\\n                        if not isinstance(passage, dict):\\n                            continue\\n                        if passage.get(\\"id\\") == group_id or passage.get(\\"group_id\\") == group_id:\\n                            matching_passage = passage\\n                            break\\n\\n                if matching_passage is None:\\n                    raise KeyError(\\n                        f\\"Question {qid!r} in {qfile} references group_id={group_id!r}, \\"\\n                        \\"but no matching top-level passage exists\\"\\n                    )\\n                if isinstance(matching_passage, dict):\\n                    passage_text = matching_passage.get(\\"text\\", matching_passage.get(\\"passage\\"))\\n                else:\\n                    passage_text = matching_passage\\n                if passage_text is None:\\n                    raise KeyError(\\n                        f\\"Question {qid!r} in {qfile} references group_id={group_id!r}, \\"\\n                        \\"but the matching top-level passage has no text\\"\\n                    )\\n                sections.append(f\\"Passage:\\\\n{render_question_value(passage_text)}\\")\\n\\n            sections.append(f\\"Question:\\\\n{render_question_value(q[\'stem\'])}\\")\\n\\n            options_section = render_numbered_section(\\"Options\\", q.get(\\"options\\"))\\n            if options_section is not None:\\n                sections.append(options_section)\\n\\n            pairs = q.get(\\"pairs\\")\\n            if isinstance(pairs, dict):\\n                for side in (\\"left\\", \\"right\\"):\\n                    pair_section = render_numbered_section(f\\"Pairs ({side})\\", pairs.get(side))\\n                    if pair_section is not None:\\n                        sections.append(pair_section)\\n            elif pairs is not None:\\n                pair_section = render_numbered_section(\\"Pairs\\", pairs)\\n                if pair_section is not None:\\n                    sections.append(pair_section)\\n\\n            items_section = render_numbered_section(\\"Items\\", q.get(\\"items\\"))\\n            if items_section is not None:\\n                sections.append(items_section)\\n\\n            return QuestionContext(\\n                query=\\"\\\\n\\\\n\\".join(sections),\\n                answer=q.get(\\"answer\\"),\\n                explanation=q.get(\\"explanation\\"),\\n            )\\n    raise KeyError(f\\"Question {qid!r} not found in {qfile}\\")\\n", "src/data/settings.py": "\\"\\"\\"Centralized environment access — import settings from here, don\'t read os.environ elsewhere.\\n\\nSecrets and the deployment endpoint come from the environment (a project-root ``.env``,\\nloaded once on import). Experiment parameters stay in the YAML configs, not here.\\n\\"\\"\\"\\n\\nimport os\\n\\nfrom dotenv import load_dotenv\\n\\n# Load the project\'s .env once. find_dotenv walks up from this file, so it works regardless\\n# of the current working directory (e.g. a notebook running from notebooks/). Real environment\\n# variables already set take precedence — load_dotenv does not override them.\\nload_dotenv()\\n\\n# OpenAI-compatible client credentials. \\"local\\" lets keyless local servers work.\\nOPENAI_API_KEY: str = os.environ.get(\\"OPENAI_API_KEY\\", \\"local\\")\\n# Endpoint; None lets the OpenAI SDK fall back to its default (the official API).\\nOPENAI_BASE_URL: str | None = os.environ.get(\\"OPENAI_BASE_URL\\")\\n\\n# Gemini judge credentials.\\nGEMINI_API_KEY: str = os.environ.get(\\"GEMINI_API_KEY\\", \\"\\")\\nGEMINI_ENDPOINT: str | None = os.environ.get(\\"GEMINI_ENDPOINT\\")\\n\\n# Rewriter credentials (separate from judge; e.g. xAI/Grok endpoint).\\nREWRITER_API_KEY: str = os.environ.get(\\"REWRITER_API_KEY\\", \\"\\")\\nREWRITER_BASE_URL: str | None = os.environ.get(\\"REWRITER_BASE_URL\\")\\n", "src/personalization/profiles.py": "\\"\\"\\"Learner persona definitions for the Simurgh RAG pipeline.\\n\\nSchema and rendered texts sourced from docs/personas.md.\\nThree personas (crammer, scholar, steady) are used in train/val;\\nnewcomer is held out for test only.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\n\\n@dataclass(frozen=True)\\nclass Profile:\\n    id: str\\n    comprehension: str  # L1–L4\\n    prior_knowledge: str  # L1–L4\\n    learning_goal: str  # L1–L4\\n    explanation_style: str  # L1–L4\\n    split: str  # \\"train\\" or \\"test\\"\\n    rendered: str  # natural-language prose injected into prompts\\n\\n\\nPERSONAS: dict[str, Profile] = {\\n    \\"crammer\\": Profile(\\n        id=\\"crammer\\",\\n        comprehension=\\"L1\\",\\n        prior_knowledge=\\"L1\\",\\n        learning_goal=\\"L1\\",\\n        explanation_style=\\"L4\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A ninth-grader who finds the textbook hard to follow and has little background \\"\\n            \\"on this topic. Mainly wants to pass the exam — give the answer and what is needed \\"\\n            \\"to score — but it must be spelled out simply, step by step, with examples.\\"\\n        ),\\n    ),\\n    \\"scholar\\": Profile(\\n        id=\\"scholar\\",\\n        comprehension=\\"L4\\",\\n        prior_knowledge=\\"L3\\",\\n        learning_goal=\\"L4\\",\\n        explanation_style=\\"L1\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A ninth-grader who reads dense material easily and has solid background on this \\"\\n            \\"topic. Wants to understand the underlying why and how, and the connections between \\"\\n            \\"ideas. Prefers a terse, high-level treatment without hand-holding or padding.\\"\\n        ),\\n    ),\\n    \\"steady\\": Profile(\\n        id=\\"steady\\",\\n        comprehension=\\"L3\\",\\n        prior_knowledge=\\"L2\\",\\n        learning_goal=\\"L2\\",\\n        explanation_style=\\"L2\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A capable ninth-grader with average background on this topic. Wants a correct \\"\\n            \\"answer with a brief justification, balanced toward exam needs. Does not need \\"\\n            \\"elaborate scaffolding, but does appreciate a one-line reason.\\"\\n        ),\\n    ),\\n    \\"newcomer\\": Profile(\\n        id=\\"newcomer\\",\\n        comprehension=\\"L3\\",\\n        prior_knowledge=\\"L1\\",\\n        learning_goal=\\"L4\\",\\n        explanation_style=\\"L4\\",\\n        split=\\"test\\",\\n        rendered=(\\n            \\"A bright ninth-grader who reads well but is new to this topic. Wants real \\"\\n            \\"understanding — the why and how, not just the answer — and needs worked examples \\"\\n            \\"to bridge the missing background.\\"\\n        ),\\n    ),\\n}\\n\\n\\ndef render_profile(persona_id: str) -> str:\\n    \\"\\"\\"Return the natural-language rendering for *persona_id*.\\"\\"\\"\\n    if persona_id not in PERSONAS:\\n        raise ValueError(f\\"Unknown persona: {persona_id!r}. Valid: {list(PERSONAS)}\\")\\n    return PERSONAS[persona_id].rendered\\n\\n\\ndef train_personas() -> list[Profile]:\\n    \\"\\"\\"Return the three training personas (excludes the test holdout).\\"\\"\\"\\n    return [p for p in PERSONAS.values() if p.split == \\"train\\"]\\n", "src/rag/embedder.py": "\\"\\"\\"Qwen3-Embedding-0.6B dense encoder wrapper.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport torch\\nfrom sentence_transformers import SentenceTransformer\\n\\n\\nclass Qwen3Embedder:\\n    \\"\\"\\"Encodes text to L2-normalised float32 dense vectors using Qwen3-Embedding.\\n\\n    Qwen3-Embedding is a decoder-based model that uses last-token pooling and\\n    supports instruction-prefixed queries for task-conditioned retrieval.\\n    sentence-transformers handles last-token pooling automatically via the model\'s\\n    config when trust_remote_code=True.\\n\\n    If *adapter_path* is set, a LoRA adapter (e.g. the ROPG-KD checkpoint from\\n    src/rl/ropg_kd.py) is loaded on top of the base model and merged into its weights.\\n    \\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model_name: str = \\"Qwen/Qwen3-Embedding-0.6B\\",\\n        device: str = \\"cpu\\",\\n        batch_size: int = 32,\\n        fp16: bool = False,\\n        adapter_path: str | None = None,\\n        max_seq_length: int | None = None,\\n    ) -> None:\\n        model_kwargs = {\\"torch_dtype\\": torch.float16} if fp16 else {}\\n        self.model = SentenceTransformer(\\n            model_name, device=device, trust_remote_code=True, model_kwargs=model_kwargs\\n        )\\n        if adapter_path is not None:\\n            from peft import PeftModel\\n\\n            # from_pretrained injects the adapter into the base model in place and\\n            # merge_and_unload folds the LoRA weights into it, so no reassignment is\\n            # needed (auto_model is a read-only property on the Transformer module).\\n            PeftModel.from_pretrained(self.model[0].auto_model, adapter_path).merge_and_unload()\\n        if max_seq_length is not None:\\n            # Must match embedder.max_seq_length in configs/train_ropg.yaml. If training\\n            # truncates at a different length than indexing, the same chunk receives two\\n            # different embeddings and the fine-tune is measured against the wrong vectors.\\n            self.model.max_seq_length = max_seq_length\\n        self.batch_size = batch_size\\n        self.dim: int = self.model.get_embedding_dimension()\\n\\n    def encode(self, texts: list[str]) -> np.ndarray:\\n        \\"\\"\\"Encode documents (no instruction prefix). Returns float32 (N, dim), L2-normalised.\\"\\"\\"\\n        vecs = self.model.encode(\\n            texts,\\n            batch_size=self.batch_size,\\n            normalize_embeddings=True,\\n            show_progress_bar=False,\\n        )\\n        return np.array(vecs, dtype=np.float32)\\n\\n    def encode_query(self, texts: list[str], instruction: str = \\"\\") -> np.ndarray:\\n        \\"\\"\\"Encode queries with an optional persona instruction prefix.\\n\\n        When *instruction* is provided the model sees:\\n            ``Instruct: {instruction}\\\\\\\\nQuery: {text}``\\n        which lets it condition the embedding on the learner profile.\\n        \\"\\"\\"\\n        if instruction:\\n            prompt = f\\"Instruct: {instruction}\\\\nQuery: \\"\\n            vecs = self.model.encode(\\n                texts,\\n                prompt=prompt,\\n                batch_size=self.batch_size,\\n                normalize_embeddings=True,\\n                show_progress_bar=False,\\n            )\\n        else:\\n            vecs = self.model.encode(\\n                texts,\\n                batch_size=self.batch_size,\\n                normalize_embeddings=True,\\n                show_progress_bar=False,\\n            )\\n        return np.array(vecs, dtype=np.float32)\\n", "src/rag/llm.py": "\\"\\"\\"Thin wrappers over the chat endpoints this project talks to.\\"\\"\\"\\n\\nimport openai\\n\\n\\nclass OpenAICompatClient:\\n    \\"\\"\\"Chat client for OpenAI-compatible routes: OpenAI, LMStudio, Ollama, vLLM, Metis.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        base_url: str | None,\\n        api_key: str,\\n        model: str,\\n        temperature: float = 0.2,\\n        max_tokens: int = 800,\\n        reasoning_effort: str | None = None,\\n    ) -> None:\\n        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)\\n        self.model = model\\n        self.temperature = temperature\\n        self.max_tokens = max_tokens\\n        self.reasoning_effort = reasoning_effort\\n\\n    def chat(self, messages: list[dict[str, str]]) -> str:\\n        \\"\\"\\"Send *messages* and return the assistant reply text.\\"\\"\\"\\n        request = {\\n            \\"model\\": self.model,\\n            \\"messages\\": messages,\\n            \\"temperature\\": self.temperature,\\n            \\"max_completion_tokens\\": self.max_tokens,\\n        }\\n        if self.reasoning_effort is not None:\\n            request[\\"reasoning_effort\\"] = self.reasoning_effort\\n\\n        response = self.client.chat.completions.create(**request)\\n        return response.choices[0].message.content or \\"\\"\\n\\n\\nclass GeminiClient:\\n    \\"\\"\\"Gemini chat client for Metis\'s native Google GenAI route.\\n\\n    Metis serves the Gemini family through Google\'s own protocol\\n    (``POST {base_url}/v1beta/models/{model}:generateContent``), not through its\\n    OpenAI-compatible route, so these models are unreachable with\\n    :class:`OpenAICompatClient`.\\n\\n    Automatic function calling is disabled: this client passes no tools, and leaving it on\\n    wraps every request in a tool-calling loop that logs a line per call.\\n    \\"\\"\\"\\n\\n    #: Levels the Gemini API accepts. Which subset a given model supports is model\\n    #: dependent, so the caller picks the value and the server has final say. Gemini 3.5\\n    #: and newer reject the older ``thinking_budget`` field outright.\\n    THINKING_LEVELS = (\\"minimal\\", \\"low\\", \\"medium\\", \\"high\\")\\n\\n    def __init__(\\n        self,\\n        base_url: str | None,\\n        api_key: str,\\n        model: str,\\n        system_instruction: str | None = None,\\n        temperature: float = 0.0,\\n        max_output_tokens: int = 256,\\n        thinking_level: str | None = \\"minimal\\",\\n        response_schema: dict | None = None,\\n    ) -> None:\\n        try:\\n            from google import genai\\n            from google.genai.types import (\\n                AutomaticFunctionCallingConfig,\\n                GenerateContentConfig,\\n                HttpOptions,\\n                ThinkingConfig,\\n            )\\n        except ImportError as exc:  # pragma: no cover - dependency guard\\n            raise RuntimeError(\\n                \\"Gemini models need the google-genai SDK. Install it with:\\\\n\\"\\n                \\"  uv pip install google-genai\\"\\n            ) from exc\\n\\n        # The SDK enum is case-insensitive and accepts unknown values, so a typo would\\n        # only surface as a server-side 400 once per call. Reject it here instead.\\n        level = (thinking_level or \\"\\").strip().lower()\\n        if level and level not in self.THINKING_LEVELS:\\n            raise ValueError(\\n                f\\"Unsupported thinking level {thinking_level!r}; \\"\\n                f\\"use one of {\', \'.join(self.THINKING_LEVELS)}, or an empty value to let \\"\\n                \\"the model choose\\"\\n            )\\n\\n        http_options = HttpOptions(base_url=base_url) if base_url else None\\n        self.client = genai.Client(api_key=api_key, http_options=http_options)\\n        self.model = model\\n        self.config = GenerateContentConfig(\\n            system_instruction=system_instruction,\\n            temperature=temperature,\\n            max_output_tokens=max_output_tokens,\\n            thinking_config=ThinkingConfig(thinking_level=level.upper()) if level else None,\\n            response_mime_type=\\"application/json\\" if response_schema else None,\\n            response_schema=response_schema,\\n            automatic_function_calling=AutomaticFunctionCallingConfig(disable=True),\\n        )\\n\\n    def generate(self, prompt: str) -> str:\\n        \\"\\"\\"Send *prompt* as a single user turn and return the reply text.\\n\\n        Returns an empty string when the model produced no text, which includes a reply\\n        truncated by ``max_output_tokens``: thinking tokens draw from the same budget, and\\n        no Gemini 3.x model lets thinking be turned off entirely.\\n        \\"\\"\\"\\n        response = self.client.models.generate_content(\\n            model=self.model,\\n            contents=prompt,\\n            config=self.config,\\n        )\\n        return response.text or \\"\\"\\n", "src/rag/rewriter.py": "\\"\\"\\"Persona-conditioned query rewriters.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import TYPE_CHECKING, Any\\n\\nif TYPE_CHECKING:\\n    from rag.llm import OpenAICompatClient\\n\\n_REWRITE_SYSTEM = (\\n    \\"You are a query rewriting assistant for a Persian educational RAG system. \\"\\n    \\"Given a learner profile and an original question, rewrite the question as a \\"\\n    \\"retrieval query that will surface the most pedagogically useful passages for \\"\\n    \\"that specific learner. \\"\\n    \\"Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. \\"\\n    \\"Keep it in Persian if the original is Persian. \\"\\n    \\"You may expand abbreviations, add prerequisite terms, or rephrase for clarity, \\"\\n    \\"but do not invent facts or change the question\'s intent.\\"\\n)\\n_DEFAULT_GENERATION = {\\"max_new_tokens\\": 224, \\"do_sample\\": False}\\n_ALLOWED_GENERATION_KEYS = {\\n    \\"max_new_tokens\\",\\n    \\"do_sample\\",\\n    \\"temperature\\",\\n    \\"top_p\\",\\n    \\"top_k\\",\\n    \\"repetition_penalty\\",\\n}\\n\\n\\ndef build_rewrite_messages(profile_rendered: str, query: str) -> list[dict[str, str]]:\\n    \\"\\"\\"Build the shared training, local-inference, and remote-baseline prompt.\\"\\"\\"\\n    if not profile_rendered.strip():\\n        raise ValueError(\\"profile_rendered must be nonempty\\")\\n    if not query.strip():\\n        raise ValueError(\\"query must be nonempty\\")\\n    user = (\\n        f\\"Learner profile: {profile_rendered}\\\\n\\\\n\\"\\n        f\\"Original question: {query}\\\\n\\\\n\\"\\n        \\"Rewritten retrieval query:\\"\\n    )\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": _REWRITE_SYSTEM},\\n        {\\"role\\": \\"user\\", \\"content\\": user},\\n    ]\\n\\n\\ndef _normalize_generation(generation: dict[str, Any] | None) -> dict[str, Any]:\\n    settings = {**_DEFAULT_GENERATION, **(generation or {})}\\n    unknown = sorted(set(settings) - _ALLOWED_GENERATION_KEYS)\\n    if unknown:\\n        raise ValueError(f\\"Unknown generation settings: {unknown}\\")\\n    max_new_tokens = settings.get(\\"max_new_tokens\\")\\n    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int) or max_new_tokens < 1:\\n        raise ValueError(\\"generation.max_new_tokens must be a positive integer\\")\\n    do_sample = settings.get(\\"do_sample\\")\\n    if not isinstance(do_sample, bool):\\n        raise ValueError(\\"generation.do_sample must be boolean\\")\\n    if not do_sample:\\n        settings.pop(\\"temperature\\", None)\\n        settings.pop(\\"top_p\\", None)\\n        settings.pop(\\"top_k\\", None)\\n    return settings\\n\\n\\ndef generate_rewrite_batch(\\n    model: Any,\\n    tokenizer: Any,\\n    profiles: list[str],\\n    queries: list[str],\\n    *,\\n    generation: dict[str, Any] | None = None,\\n) -> list[str]:\\n    \\"\\"\\"Generate response-only rewrites for a batch of profile/query pairs.\\"\\"\\"\\n    if not profiles or len(profiles) != len(queries):\\n        raise ValueError(\\"profiles and queries must be nonempty lists of equal length\\")\\n    settings = _normalize_generation(generation)\\n    prompt_texts = [\\n        tokenizer.apply_chat_template(\\n            build_rewrite_messages(profile, query),\\n            tokenize=False,\\n            add_generation_prompt=True,\\n            enable_thinking=False,\\n        )\\n        for profile, query in zip(profiles, queries, strict=True)\\n    ]\\n    previous_padding_side = tokenizer.padding_side\\n    tokenizer.padding_side = \\"left\\"\\n    try:\\n        inputs = tokenizer(prompt_texts, return_tensors=\\"pt\\", padding=True)\\n    finally:\\n        tokenizer.padding_side = previous_padding_side\\n    inputs = inputs.to(model.device)\\n    padded_input_length = inputs[\\"input_ids\\"].shape[1]\\n    outputs = model.generate(\\n        **inputs,\\n        **settings,\\n        pad_token_id=tokenizer.pad_token_id,\\n        eos_token_id=tokenizer.eos_token_id,\\n    )\\n    completion_ids = outputs[:, padded_input_length:]\\n    rewrites = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)\\n    cleaned = [rewrite.strip() for rewrite in rewrites]\\n    empty_indices = [index for index, rewrite in enumerate(cleaned) if not rewrite]\\n    if empty_indices:\\n        raise RuntimeError(f\\"Model returned empty rewrites at batch indices {empty_indices}\\")\\n    return cleaned\\n\\n\\nclass PromptedRewriter:\\n    \\"\\"\\"Call a remote LLM for a persona-conditioned query rewrite.\\"\\"\\"\\n\\n    def __init__(self, llm: OpenAICompatClient) -> None:\\n        self.llm = llm\\n\\n    def rewrite(self, profile_rendered: str, query: str) -> str:\\n        messages = build_rewrite_messages(profile_rendered, query)\\n        rewrite = self.llm.chat(messages).strip()\\n        if not rewrite:\\n            raise RuntimeError(\\"Remote rewriter returned an empty completion\\")\\n        return rewrite\\n\\n\\nclass DPORewriter:\\n    \\"\\"\\"Run a Qwen3 base model or a promoted DPO-family LoRA adapter.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model_name: str = \\"Qwen/Qwen3-4B\\",\\n        adapter_path: str | None = None,\\n        device: str = \\"cuda\\",\\n        max_seq_length: int = 768,\\n        generation: dict[str, Any] | None = None,\\n    ) -> None:\\n        import torch\\n        from unsloth import FastLanguageModel\\n\\n        requested_device = torch.device(device)\\n        loader_kwargs: dict[str, Any] = {}\\n        if requested_device.type == \\"cuda\\":\\n            if not torch.cuda.is_available():\\n                raise RuntimeError(f\\"Requested {device}, but CUDA is unavailable\\")\\n            device_index = requested_device.index if requested_device.index is not None else 0\\n            torch.cuda.set_device(device_index)\\n            resolved_device = torch.device(\\"cuda\\", device_index)\\n            load_in_4bit = True\\n            # One whole replica on one GPU. Left to itself, Unsloth\'s planner spreads the\\n            # model over every visible GPU and pins the tied embedding to the output head\'s\\n            # device, so generation feeds token ids on one GPU into an embedding on another.\\n            loader_kwargs[\\"device_map\\"] = {\\"\\": device_index}\\n        elif requested_device.type == \\"cpu\\":\\n            resolved_device = torch.device(\\"cpu\\")\\n            load_in_4bit = False\\n        else:\\n            raise ValueError(f\\"Unsupported DPO rewriter device: {device!r}\\")\\n\\n        normalized_generation = _normalize_generation(generation)\\n        model, tokenizer = FastLanguageModel.from_pretrained(\\n            model_name=model_name,\\n            load_in_4bit=load_in_4bit,\\n            max_seq_length=max_seq_length,\\n            **loader_kwargs,\\n        )\\n        if tokenizer.pad_token is None:\\n            tokenizer.pad_token = tokenizer.eos_token\\n        if adapter_path is not None:\\n            from peft import PeftModel\\n\\n            model = PeftModel.from_pretrained(model, adapter_path)\\n        if resolved_device.type == \\"cpu\\":\\n            model = model.to(resolved_device)\\n        FastLanguageModel.for_inference(model)\\n\\n        placements = {parameter.device for parameter in model.parameters()}\\n        if placements != {resolved_device}:\\n            raise RuntimeError(\\n                f\\"DPO rewriter must hold one replica on {resolved_device}, but its weights \\"\\n                \\"are spread over \\" + \\", \\".join(sorted(str(place) for place in placements))\\n            )\\n        self.model_name = model_name\\n        self.adapter_path = adapter_path\\n        self.max_seq_length = max_seq_length\\n        self.generation = normalized_generation\\n        self.model = model\\n        self.tokenizer = tokenizer\\n\\n    def rewrite_batch(self, profiles: list[str], queries: list[str]) -> list[str]:\\n        return generate_rewrite_batch(\\n            self.model,\\n            self.tokenizer,\\n            profiles,\\n            queries,\\n            generation=self.generation,\\n        )\\n\\n    def rewrite(self, profile_rendered: str, query: str) -> str:\\n        return self.rewrite_batch([profile_rendered], [query])[0]\\n"}')
for relative_path, content in SOURCE_FILES.items():
    path = WORKDIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
for package in ("data", "rag", "personalization"):
    (WORKDIR / "src" / package / "__init__.py").touch()

CONFIG_YAML = '# Held-out ablation: five arms on the held-out question split, all four personas.\ndata:\n  questions_dir: data/questions\n  test_qids_path: data/splits/test_qids.txt\n  corpus_path: data/chunks/corpus.jsonl\noutput_dir: results/ablation\n# All four profiles in personalization.profiles.PERSONAS. `newcomer` is the persona held out\n# of training; the other three are seen personas on unseen questions. Drop entries here to\n# shrink the run — every expected count is derived from this list.\npersonas: [crammer, newcomer, scholar, steady]\n# Repetitions of the identical remote calls. Not seeds: no endpoint here exposes a verified\n# deterministic seed contract, so these measure endpoint sampling variance only. One\n# replicate means the run makes no variance claim. Widening to [1, 2, 3] triples the remote\n# bill and needs no code change; the notebook exposes REPLICATES_OVERRIDE for the same edit.\nreplicates: [1]\n\nembedder:\n  model_name: Qwen/Qwen3-Embedding-0.6B\n  # Must equal embedder.max_seq_length in configs/train_ropg.yaml.\n  max_seq_length: 2048\n  fp16: true\n  batch_size: 16\n\nrewriter:\n  model_name: Qwen/Qwen3-4B\n  max_seq_length: 768\n  batch_size: 8\n  generation:\n    max_new_tokens: 224\n    do_sample: false\n\nartifacts:\n  ropg_adapter_path: models/ropg/runB/checkpoint-best\n  dpo_adapter_path: models/dpo/dpo/dpo_best\n\n# Baseline ladder, one rung per arm. `profile: false` strips the learner profile from both\n# the retrieval instruction and the answer prompt; the judge always receives it.\narms:\n  - {name: base_embedder_no_profile,       embedder: base,  profile: false, rewriter: none}\n  - {name: base_embedder,                  embedder: base,  profile: true,  rewriter: none}\n  - {name: trained_embedder,               embedder: ropg,  profile: true,  rewriter: none}\n  - {name: trained_embedder_base_rewriter, embedder: ropg,  profile: true,  rewriter: base}\n  - {name: trained_embedder_rewriter,      embedder: ropg,  profile: true,  rewriter: dpo}\n\nretrieval:\n  top_k: 5\n\ngenerator:\n  model: deepseek-v4-flash\n  temperature: 0.2\n  max_completion_tokens: 1024\n  retry: {max_attempts: 3, initial_backoff_seconds: 1.0, backoff_multiplier: 2.0, max_backoff_seconds: 8.0}\n\njudges:\n  # Primary. Also labelled the DPO preference pairs, so arm-5 numbers are not judge-independent\n  # on this judge alone — that is what the secondary judge is for.\n  primary:\n    model: gpt-5.6-luna\n    reasoning_effort: low\n    temperature: 1.0\n    # Covers hidden low-effort reasoning plus the small JSON reply.\n    max_completion_tokens: 2048\n    retry: {max_attempts: 3, initial_backoff_seconds: 1.0, backoff_multiplier: 2.0, max_backoff_seconds: 8.0}\n  # Secondary. Independent of the pair labeller and of the ROPG teacher.\n  secondary:\n    model: gemini-3.7-flash\n    thinking_level: minimal\n    temperature: 0.0\n    max_output_tokens: 512\n    retry: {max_attempts: 3, initial_backoff_seconds: 1.0, backoff_multiplier: 2.0, max_backoff_seconds: 8.0}\n\nexecution:\n  # Concurrent in-flight remote requests per rank; 16 across two T4s.\n  max_workers: 8\n  bootstrap_samples: 5000\n  bootstrap_seed: 42\n'
print(f"Wrote {len(SOURCE_FILES)} source files under {WORKDIR}")


## Adapter discovery

In [ ]:
import json


def _candidates(leaf_name):
    found = []
    for config_path in sorted(Path("/kaggle/input").rglob(f"{leaf_name}/adapter_config.json")):
        try:
            base = json.loads(config_path.read_text(encoding="utf-8")).get(
                "base_model_name_or_path", ""
            )
        except json.JSONDecodeError:
            base = "<unreadable adapter_config.json>"
        found.append((config_path.parent, base))
    return found


def discover(leaf_name, base_suffix, require_segment=None, forbid_segments=()):
    """Return the single adapter directory whose recorded base model matches."""
    found = _candidates(leaf_name)
    validated = []
    for directory, base in found:
        parts = set(directory.parts)
        if require_segment is not None and require_segment not in parts:
            continue
        if parts & set(forbid_segments):
            continue
        if str(base).endswith(base_suffix):
            validated.append(directory)
    if len(validated) != 1:
        for directory, base in found:
            print(f"  candidate {directory} -> {base}")
        raise RuntimeError(
            f"Expected exactly one {leaf_name} adapter on {base_suffix}, found "
            f"{len(validated)}: {validated}"
        )
    return validated[0]


if ROPG_ADAPTER_DIR:
    ROPG_ADAPTER = Path(ROPG_ADAPTER_DIR)
else:
    ROPG_ADAPTER = discover("checkpoint-best", "Qwen3-Embedding-0.6B")
# The DPO family trained three arms into sibling directories. Only the plain `dpo` arm is
# the promoted rewriter; wpo and robust_dpo carry the same leaf name and the same base model.
if DPO_ADAPTER_DIR:
    DPO_ADAPTER = Path(DPO_ADAPTER_DIR)
else:
    DPO_ADAPTER = discover(
        "dpo_best", "Qwen3-4B", require_segment="dpo", forbid_segments=("wpo", "robust_dpo")
    )
print("ROPG adapter:", ROPG_ADAPTER)
print("DPO adapter: ", DPO_ADAPTER)


## Resolve the runtime config

In [ ]:
import yaml

config = yaml.safe_load(CONFIG_YAML)
config["data"]["questions_dir"] = str(DATA_ROOT / "questions")
config["data"]["test_qids_path"] = str(DATA_ROOT / "splits" / "test_qids.txt")
config["data"]["corpus_path"] = str(DATA_ROOT / "chunks" / "corpus.jsonl")
config["artifacts"]["ropg_adapter_path"] = str(ROPG_ADAPTER)
config["artifacts"]["dpo_adapter_path"] = str(DPO_ADAPTER)
config["output_dir"] = str(OUTPUT_ROOT / "ablation")
if REPLICATES_OVERRIDE is not None:
    config["replicates"] = list(REPLICATES_OVERRIDE)
if PERSONAS_OVERRIDE is not None:
    config["personas"] = list(PERSONAS_OVERRIDE)
if MAX_WORKERS_OVERRIDE is not None:
    config["execution"]["max_workers"] = int(MAX_WORKERS_OVERRIDE)

FULL_CONFIG_PATH = WORKDIR / "eval_ablation_runtime.yaml"
FULL_CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

# A separate output directory, not just a smaller config: sharing one would let two-question
# smoke rows resume into the full run and satisfy its expected-key check.
smoke_config = yaml.safe_load(yaml.safe_dump(config))
smoke_config["output_dir"] = str(OUTPUT_ROOT / "ablation_smoke")
smoke_config["replicates"] = [config["replicates"][0]]
SMOKE_CONFIG_PATH = WORKDIR / "eval_ablation_smoke.yaml"
SMOKE_CONFIG_PATH.write_text(yaml.safe_dump(smoke_config, sort_keys=False), encoding="utf-8")

QUESTION_REFS = [
    line.strip()
    for line in Path(config["data"]["test_qids_path"]).read_text(encoding="utf-8").splitlines()
    if line.strip()
]


def expected_keys(resolved, n_questions):
    """The one place a row count is computed. Never hard-code it in an assertion."""
    return (
        n_questions
        * len(resolved["personas"])
        * len(resolved["arms"])
        * len(resolved["replicates"])
    )


FULL_EXPECTED = expected_keys(config, len(QUESTION_REFS))
SMOKE_EXPECTED = expected_keys(smoke_config, min(SMOKE_LIMIT, len(QUESTION_REFS)))
print("Questions: ", len(QUESTION_REFS))
print("Personas:  ", config["personas"])
print("Replicates:", config["replicates"])
print("Expected keys — full:", FULL_EXPECTED, "| smoke:", SMOKE_EXPECTED)


## Preflight — no CUDA, no model weights

In [ ]:
import subprocess
import sys

import torch

n_gpus = torch.cuda.device_count()
print("Visible GPUs:", n_gpus)
for index in range(n_gpus):
    print(f"  cuda:{index} {torch.cuda.get_device_name(index)}")
assert n_gpus == 2, f"This notebook launches two ranks; got {n_gpus} GPUs"

subprocess.run(
    [
        sys.executable,
        "benchmarks/eval_runner.py",
        "--config",
        str(FULL_CONFIG_PATH),
        "--preflight-only",
    ],
    check=True,
    env=os.environ.copy(),
)
print("Derived expected keys:", FULL_EXPECTED)


## Two-question smoke, both ranks

In [ ]:
import subprocess

ARM_NAMES = [arm["name"] for arm in config["arms"]]


def load_results(resolved):
    path = Path(resolved["output_dir"]) / "results.jsonl"
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def check_rows(rows, resolved, expected_count):
    """Every invariant a run must satisfy before its numbers mean anything."""
    successful = [row for row in rows if row["status"] == "ok"]
    assert len(successful) == expected_count, (
        f"expected {expected_count} successful rows, got {len(successful)} "
        f"of {len(rows)} merged"
    )
    assert {row["arm"] for row in successful} == set(ARM_NAMES)
    assert {row["persona_id"] for row in successful} == set(resolved["personas"])
    assert {row["rank"] for row in rows} == {0, 1}, "one rank produced nothing"
    generator = resolved["generator"]["model"]
    assert all(row["generator_model"] == generator for row in successful)
    for row in successful:
        for role in ("primary", "secondary"):
            scores = row[f"{role}_scores"]
            if scores is None:
                continue
            assert scores["answer_correctness"] in (0, 1), scores
            for metric in (
                "context_utility",
                "faithfulness",
                "persona_alignment",
                "pedagogical_quality",
            ):
                assert 0 <= scores[metric] <= 4, scores
    scored_twice = sum(1 for row in successful if row["secondary_scores"] is not None)
    assert scored_twice >= 0.9 * expected_count, (
        f"only {scored_twice}/{expected_count} rows carry a secondary score; "
        "judge agreement would be computed on a biased subset"
    )
    # Retrieval sanity. With no profile in the query, four personas submit identical text
    # and must retrieve identical chunks. With a profile, at least one question must
    # retrieve differently — if none does, the instruction never reached encode_query.
    by_arm_question = {}
    for row in successful:
        chunk_ids = tuple(hit["chunk_id"] for hit in row["hits"])
        by_arm_question.setdefault((row["arm"], row["question_ref"]), {})[
            row["persona_id"]
        ] = chunk_ids
    if len(resolved["personas"]) > 1:
        unprofiled = [
            set(personas.values())
            for (arm, _q), personas in by_arm_question.items()
            if arm == "base_embedder_no_profile"
        ]
        assert all(len(variants) == 1 for variants in unprofiled), (
            "the unprofiled arm retrieved differently per persona"
        )
        profiled = [
            set(personas.values())
            for (arm, _q), personas in by_arm_question.items()
            if arm == "base_embedder"
        ]
        assert any(len(variants) > 1 for variants in profiled), (
            "the persona instruction changed no retrieval anywhere"
        )
    print(f"{len(successful)} successful rows, {scored_twice} with both judges")


smoke_command = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "--tee",
    "3",
    "--log-dir",
    str(OUTPUT_ROOT / "smoke_torchrun_logs"),
    "benchmarks/eval_runner.py",
    "--config",
    str(SMOKE_CONFIG_PATH),
    "--limit",
    str(SMOKE_LIMIT),
]
try:
    subprocess.run(smoke_command, check=True, env=os.environ.copy())
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        f"Smoke run failed (exit {error.returncode}). Read the per-rank logs under "
        f"{OUTPUT_ROOT / 'smoke_torchrun_logs'} before launching the full run."
    ) from error

check_rows(load_results(smoke_config), smoke_config, SMOKE_EXPECTED)


## Full run

In [ ]:
import subprocess

full_command = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "--tee",
    "3",
    "--log-dir",
    str(OUTPUT_ROOT / "torchrun_logs"),
    "benchmarks/eval_runner.py",
    "--config",
    str(FULL_CONFIG_PATH),
]
try:
    subprocess.run(full_command, check=True, env=os.environ.copy())
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        f"Full run failed (exit {error.returncode}). Re-running this cell resumes: every "
        "key already scored under this config is skipped and only failures are retried."
    ) from error

check_rows(load_results(config), config, FULL_EXPECTED)


## Inspect the outputs

In [ ]:
import csv

output_dir = Path(config["output_dir"])
manifest = json.loads((output_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("expected/merged/successful:", manifest["expected_keys"], manifest["merged_keys"],
      manifest["successful_keys"])
print("failures by stage:", manifest["failures_by_stage"])
print("rows missing a secondary score:", manifest["secondary_judge_missing"])
print("stale records ignored:", manifest["stale_records"])


def show(name, keep=lambda row: True, limit=60):
    print("=" * 78)
    print(name)
    with (output_dir / name).open(encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if keep(row)]
    for row in rows[:limit]:
        print("  " + " | ".join(f"{key}={value}" for key, value in row.items()))
    print(f"  ({len(rows)} rows shown of the filtered set)")


# Pooled first, then the split that matters: a gain on the three seen personas that vanishes
# on the held-out `newcomer` is persona overfitting, not a retrieval result.
show("summary.csv", lambda row: row["persona"] in ("all", "all_train", "all_test"))
show("paired_deltas.csv", lambda row: row["judge"] == "primary")
show("judge_agreement.csv")


## If something failed

* **A key failed to generate or to be judged.** Re-run the full-run cell. Resume keys on
  `config_sha256`, so every row already scored under this exact config is skipped and only
  failures are retried. Changing the config invalidates the old rows rather than silently
  mixing them: they are excluded and counted as `stale_records`.
* **Sustained 429s.** Set `MAX_WORKERS_OVERRIDE = 4` in the knobs cell and re-run. Nothing
  already completed is lost.
* **Adapter discovery found zero or several candidates.** It prints every candidate with the
  base model each one records. Pick one and set `ROPG_ADAPTER_DIR` or `DPO_ADAPTER_DIR`.
* **A judge disagrees with the other on the sign of a delta.** That is a result about the
  measurement, not the model. Repo measurement puts judge agreement near 70% on decisive
  verdicts, so a delta below that noise floor is not claimable from either judge.


## Export

In [ ]:
import shutil

archive = shutil.make_archive(
    str(Path("/kaggle/working") / "eval-ablation"), "zip", root_dir=OUTPUT_ROOT
)
print("Created:", archive)
